
### Optical Flow、Point Tracking 与 SIFT

### 1. Optical Flow（光流）

#### 1.1 光流的定义

光流（Optical Flow）描述图像中像素或局部区域随时间变化产生的二维位移。

假设连续两帧图像分别为：

$$
I(x,y,t), \qquad I(x,y,t+\Delta t)
$$

如果某个点在第一帧中的位置为 $(x,y)$，那么它在下一帧中移动到了：

$$
(x+u\Delta t,\ y+v\Delta t)
$$

其中：

- $u$：水平方向的速度。
- $v$：竖直方向的速度。
- $(u,v)$：该点的光流速度向量。

如果直接用两帧之间的位移表示光流，则可以省略 $\Delta t$，写成 $(x+u,y+v)$。

光流场可以表示为：

$$
\mathbf{w}(x,y)=
\begin{bmatrix}
u(x,y)\\
v(x,y)
\end{bmatrix}
$$

根据估计范围，可以分为：

- **稠密光流（Dense Optical Flow）**：估计图像中几乎每个像素的运动。
- **稀疏光流（Sparse Optical Flow）**：只估计选定关键点的运动。

光流描述的是图像平面中的二维运动，不直接等同于真实世界中的三维运动。物体运动、相机运动、深度变化等因素都可能影响光流。

#### 1.2 Brightness Constancy（亮度恒常假设）

传统光流方法通常假设：一个点在短时间内移动时，其图像亮度近似不变。

因此：

$$
I(x+u\Delta t,\ y+v\Delta t,\ t+\Delta t)
\approx I(x,y,t)
$$

这个假设并不总是成立，例如：

- 光照发生变化；
- 物体发生反射或遮挡；
- 物体外观随时间变化；
- 运动幅度过大，导致局部匹配困难。

但它提供了一个建立光流约束的起点。

#### 1.3 Optical Flow Constraint Equation（光流约束方程）

对亮度恒常公式进行一阶 Taylor 展开：

$$
\begin{aligned}
&I(x+u\Delta t,y+v\Delta t,t+\Delta t)\\
&\approx I(x,y,t)
+I_xu\Delta t
+I_yv\Delta t
+I_t\Delta t
\end{aligned}
$$

其中：

$$
I_x=\frac{\partial I}{\partial x},\qquad
I_y=\frac{\partial I}{\partial y},\qquad
I_t=\frac{\partial I}{\partial t}
$$

代入亮度恒常假设，消去两边相同的 $I(x,y,t)$，并除以 $\Delta t$，得到：

$$
\boxed{I_xu+I_yv+I_t=0}
$$

该方程称为光流约束方程（Optical Flow Constraint Equation）。

它将图像的时空梯度与运动联系起来。

**核心问题：** 一个方程包含两个未知数 $u,v$，因此单独依靠该方程，通常无法唯一确定光流。

#### 1.4 Aperture Problem（孔径问题）

孔径问题说明：只观察局部图像时，可能无法确定物体运动的完整方向。

例如，观察一条竖直边缘：

- 图像的水平方向有明显梯度；
- 图像的竖直方向梯度可能接近零。

此时光流约束方程近似为：

$$
I_xu+I_t=0
$$

它能够约束水平方向的运动，却不能充分确定竖直方向的运动。

直观上，我们可以判断边缘沿垂直于边缘的方向移动了多少，却很难只凭这条边缘判断沿边缘方向的运动。

解决孔径问题的一种思路是引入更多约束，例如假设一个局部区域内的运动相同。

#### 1.5 Lucas–Kanade（LK）光流

Lucas–Kanade 方法假设：在一个局部窗口内，所有像素具有近似相同的光流。

对于窗口内的多个像素，可以得到多个约束：

$$
I_{x_i}u+I_{y_i}v=-I_{t_i}
$$

将它们写成矩阵形式：

$$
A\mathbf{w}=\mathbf{b}
$$

其中：

$$
A=
\begin{bmatrix}
I_{x_1}&I_{y_1}\\
I_{x_2}&I_{y_2}\\
\vdots&\vdots\\
I_{x_n}&I_{y_n}
\end{bmatrix}
$$

$$
\mathbf{w}=
\begin{bmatrix}
u\\v
\end{bmatrix},
\qquad
\mathbf{b}=
-\begin{bmatrix}
I_{t_1}\\
I_{t_2}\\
\vdots\\
I_{t_n}
\end{bmatrix}
$$

使用最小二乘法求解：

$$
\hat{\mathbf{w}}
=\arg\min_{\mathbf{w}}\|A\mathbf{w}-\mathbf{b}\|_2^2
$$

如果 $A^\top A$ 可逆，则：

$$
\boxed{
\hat{\mathbf{w}}=(A^\top A)^{-1}A^\top\mathbf{b}
}
$$

实际实现中一般使用数值求解方法，而不是显式计算矩阵逆。

LK 的核心是：利用局部区域内的多个像素提供约束，共同估计一个运动向量。

#### 1.6 结构张量与角点

LK 方法能否稳定估计运动，与局部图像的纹理结构密切相关。

定义结构张量：

$$
G=A^\top A
=
\begin{bmatrix}
\sum I_x^2&\sum I_xI_y\\
\sum I_xI_y&\sum I_y^2
\end{bmatrix}
$$

设其两个特征值为 $\lambda_1,\lambda_2$。

- 两个特征值都较大：两个方向都有明显纹理，通常容易稳定跟踪。
- 一个较大、一个较小：局部接近一条边缘，某个运动方向不容易确定。
- 两个都很小：局部纹理较弱，运动估计容易受到噪声影响。

因此，角点通常比单纯的边缘点更适合进行稳定的特征跟踪。

#### 1.7 多尺度光流（Image Pyramids）

传统局部光流方法通常依赖较小的运动假设。当运动幅度过大时，直接在原图上进行局部匹配可能失败。

多尺度方法将图像逐渐下采样，形成图像金字塔：

$$
I^0,I^1,\ldots,I^L
$$

其中较高层级对应更低的空间分辨率。

在低分辨率图像中，同样的实际位移对应更少的像素，因此更容易被局部方法处理。

典型流程：

1. 在低分辨率图像上估计较大范围的运动。
2. 将光流上采样到更高分辨率。
3. 使用上一层的估计作为初始化。
4. 在当前分辨率下进一步修正光流。
5. 重复上述过程，直到原始分辨率。

注意：下采样会缩小像素位移，但上采样光流时也必须正确调整位移数值，不能只改变张量的空间尺寸。

---

### 2. RAFT：基于相关性与循环更新的光流估计

#### 2.1 RAFT 的核心思想

RAFT（Recurrent All-Pairs Field Transforms）是一种深度学习光流方法。

它的关键思想是：

1. 提取两帧图像的特征。
2. 计算两帧特征之间的全对相关性。
3. 根据当前光流估计查询相关性。
4. 使用循环更新模块逐步修正光流。
5. 将最终低分辨率光流上采样到原图尺寸。

与传统 LK 方法不同，RAFT 不直接通过亮度梯度方程求解光流，而是利用学习得到的特征和相关性来寻找对应位置。

#### 2.2 特征提取

设输入两帧图像为 $I_1,I_2$。

经过特征编码器后，得到：

$$
F_1,F_2\in\mathbb{R}^{H'\times W'\times D}
$$

其中：

- $H',W'$：特征图的空间尺寸。
- $D$：每个位置的特征维度。

每个空间位置都对应一个特征向量，而不再只是原始像素颜色。

#### 2.3 All-Pairs Correlation（全对相关性）

RAFT 计算第一帧每个特征位置与第二帧每个特征位置之间的相似度：

$$
C(p,q)=F_1(p)^\top F_2(q)
$$

其中：

- $p$：第一帧中的特征位置。
- $q$：第二帧中的候选匹配位置。
- $C(p,q)$：两个位置的特征相关性。

如果第一帧有 $H'W'$ 个位置，第二帧也有 $H'W'$ 个位置，那么相关性体包含所有位置对的匹配分数。

其概念形状为：

$$
H'\times W'\times H'\times W'
$$

这意味着每个第一帧位置，都可以查询第二帧中的任意候选位置。

全对相关性提供了广泛的匹配证据，但直接处理所有位置对会带来显著的内存和计算开销。

#### 2.4 Correlation Pyramid（相关性金字塔）

RAFT 对相关性体的目标图像空间维度进行池化，构建多个尺度的相关性表示。

直观上：

- 较高分辨率的层级提供精细的局部匹配信息。
- 较低分辨率的层级提供更大范围的匹配上下文。

一个重要区别是：RAFT 不是每次迭代都重新计算全对相关性。相关性体和相关性金字塔在前面构建好，后续迭代反复查询已有的相关性信息。

#### 2.5 根据当前光流查询局部相关性

设第一帧某个位置为 $p$，当前光流估计为 $f(p)$，那么当前对应位置为：

$$
q=p+f(p)
$$

RAFT 围绕这个位置，在不同层级的相关性图中查询局部邻域。

尽管初始相关性包含全局位置对的信息，每次更新时仍然可以聚焦于当前估计附近的候选位置。

这兼顾了全局匹配证据和局部迭代更新。

#### 2.6 ConvGRU 与迭代更新

RAFT 使用循环更新模块（通常基于 ConvGRU）综合当前状态、相关性信息等内容，预测光流增量。

更新过程可以写为：

$$
f_{k+1}=f_k+\Delta f_k
$$

其中：

- $f_k$：第 $k$ 次迭代的光流估计。
- $\Delta f_k$：本轮预测的修正量。
- $f_{k+1}$：更新后的光流。

这个过程类似逐步修正答案：

1. 根据当前估计查找匹配证据。
2. 判断当前位置可能需要怎样调整。
3. 预测一个位移增量。
4. 更新光流。
5. 重复迭代。

RAFT 可以在多个迭代阶段产生光流预测，并通过序列损失对中间预测进行监督。

#### 2.7 上采样

RAFT 通常先在较低分辨率的特征网格上估计光流，最后将结果恢复到输入图像的分辨率。

它使用学习得到的凸组合上采样机制，而不只是简单地插值。

这样可以利用局部信息更好地恢复细节，尤其是在运动边界附近。

#### 2.8 RAFT 与传统光流的区别

| 比较维度 | Lucas–Kanade | RAFT |
|---|---|---|
| 核心依据 | 亮度恒常与局部运动一致性 | 学习到的特征与相关性 |
| 求解方式 | 局部最小二乘 | 相关性查询与循环更新 |
| 运动范围 | 经典局部版本对大位移较敏感 | 通过全对相关性等机制处理较大位移 |
| 训练需求 | 经典形式不需要深度学习训练 | 需要训练数据和学习到的参数 |
| 输出 | 通常为选定点的局部运动 | 通常为稠密光流场 |

二者解决的是相同类型的问题，但使用了不同的建模与求解方式。

---

### 3. Point Tracking（点跟踪）

#### 3.1 点跟踪的定义

Point Tracking 关注的是：给定视频中的一个或多个点，持续确定它们在后续帧中的位置。

假设视频包含 $T$ 帧：

$$
I_1,I_2,\ldots,I_T
$$

对于第 $i$ 个被跟踪的点，可以将它的轨迹表示为：

$$
\mathcal{T}_i=
\{(x_i^t,y_i^t)\}_{t=1}^{T}
$$

现实中，某个点可能会离开画面、被遮挡，或者变得无法可靠匹配，因此实际跟踪还需要考虑可见性。

#### 3.2 Optical Flow 与 Point Tracking 的区别

**Optical Flow：**

- 关注两帧图像之间的二维位移。
- 通常估计稠密光流场。
- 直接回答每个位置在下一帧中移动了多少。

**Point Tracking：**

- 关注选定点在整个视频中的轨迹。
- 需要跨越多帧保持点的身份一致性。
- 需要处理遮挡、重新出现和长期匹配问题。

如果已经获得光流场，可以用它传播点的位置：

$$
\mathbf{p}_{t+1}
=
\mathbf{p}_t+\mathbf{w}_t(\mathbf{p}_t)
$$

其中 $\mathbf{w}_t(\mathbf{p}_t)$ 是当前帧中该点位置处的光流。

但是，连续传播可能产生误差累积。

#### 3.3 为什么连续传播会漂移？

假设每一帧的位移估计都有少量误差：

$$
\hat{\mathbf{w}}_t=\mathbf{w}_t+\boldsymbol{\epsilon}_t
$$

那么点的位置会不断受到误差影响。

在简化情况下，连续传播后的误差可表示为：

$$
\mathbf{e}_T\approx
\sum_{t=1}^{T-1}\boldsymbol{\epsilon}_t
$$

这说明即使每一帧的估计都比较准确，长期跟踪也可能逐渐偏离真实轨迹。

真实模型中的误差还可能相关、随位置变化，并通过后续估计进一步放大。

#### 3.4 点跟踪的主要挑战

- **遮挡（Occlusion）**：点暂时被其他物体挡住。
- **重新出现（Reappearance）**：点在被遮挡后重新可见。
- **外观变化**：光照、尺度、视角或形变发生变化。
- **长期漂移**：预测误差不断积累。
- **身份保持**：多个相似点之间容易发生混淆。
- **可见性判断**：需要判断某个点在某帧中是否真实可见。

这些问题说明，点跟踪不只是估计相邻两帧的位移，还需要维护跨时间的对应关系。

---

### 4. TAPIR：长期点跟踪

#### 4.1 TAPIR 的基本思想

TAPIR 是一种面向视频中点轨迹的跟踪方法。它将逐帧匹配与时间维度上的轨迹优化结合起来。

从概念上，可以把它理解为两个相互配合的阶段：

1. **逐帧匹配**：针对查询点，在视频各帧中寻找可能的对应位置。
2. **时间维度上的细化**：利用轨迹上下文进一步修正位置，并估计可见性。

这样做的目的，是避免完全依赖相邻帧之间的位移逐步累积。

#### 4.2 与单纯光流传播的区别

单纯使用光流传播时：

$$
\mathbf{p}_{t+1}
=
\mathbf{p}_t+\mathbf{w}_t(\mathbf{p}_t)
$$

如果中间某一步发生错误，后续位置可能继续受到影响。

TAPIR 一类方法则会利用不同帧的匹配证据和时间信息来优化轨迹，而不是只沿着先前预测的位置不断向前累加。

因此，它更适合研究长期点跟踪问题。

#### 4.3 可见性的重要性

跟踪系统不仅需要输出点的位置，还需要判断这个点在当前帧是否可见。

如果一个点被遮挡，那么当前帧中的预测位置不一定具有可靠的图像对应。

因此，位置估计与可见性判断应当结合起来理解：

- 点可见时，可以根据图像证据进行匹配。
- 点被遮挡时，系统需要降低对当前观测的信任。
- 点重新出现时，需要利用新的图像证据恢复匹配。

需要注意：不同点跟踪方法的网络结构和实现细节可能不同。这里重点是理解 TAPIR 的设计动机，而不是把它简单等同于某一种固定的光流算法。

---

### 5. Scene Flow（三维场景流）

#### 5.1 从二维光流到三维运动

光流描述的是图像平面中的二维位移，而 Scene Flow 描述场景点在三维空间中的运动。

设某个场景点在时刻 $t$ 的三维位置为：

$$
\mathbf{P}_t=
\begin{bmatrix}
X_t\\Y_t\\Z_t
\end{bmatrix}
$$

在下一时刻的位置为：

$$
\mathbf{P}_{t+1}=
\begin{bmatrix}
X_{t+1}\\Y_{t+1}\\Z_{t+1}
\end{bmatrix}
$$

则三维场景流为：

$$
\boxed{
\mathbf{s}_t(\mathbf{P})=
\mathbf{P}_{t+1}-\mathbf{P}_t
}
$$

也就是：

$$
\mathbf{s}_t=
\begin{bmatrix}
\Delta X\\
\Delta Y\\
\Delta Z
\end{bmatrix}
$$

与二维光流相比，Scene Flow 还包含深度方向的运动信息。

#### 5.2 为什么二维光流不能直接恢复三维运动？

针孔相机的投影关系为：

$$
x=f\frac{X}{Z},\qquad
y=f\frac{Y}{Z}
$$

因此，像素位置的变化不仅取决于 $X,Y$ 的变化，也取决于深度 $Z$ 的变化。

对连续时间进行微分：

$$
\dot{x}
=
f\frac{\dot X Z-X\dot Z}{Z^2}
$$

$$
\dot{y}
=
f\frac{\dot Y Z-Y\dot Z}{Z^2}
$$

这说明二维图像运动是三维运动经过透视投影后的结果。

不同的三维运动可能产生相似的二维光流，因此仅凭二维光流通常不能唯一恢复完整的三维运动。

#### 5.3 Scene Flow 的应用

Scene Flow 可以帮助理解：

- 自动驾驶中物体的三维运动；
- 机器人对动态环境的感知；
- 场景中的运动分割；
- 动态三维重建；
- 运动物体的深度变化。

核心区别：

| 方法 | 主要估计对象 |
|---|---|
| Optical Flow | 图像平面中的二维位移 |
| Point Tracking | 指定点跨帧的轨迹 |
| Scene Flow | 场景点的三维位移 |

它们可以相互配合，但不能互相简单替代。

---

### 6. FlowMap：从图像对应关系恢复场景几何

#### 6.1 核心问题

多视图三维重建希望从不同视角的图像中恢复相机运动与场景结构。

已知图像之间的对应点或光流后，我们可以利用相机投影模型建立几何约束。

对于静态场景，相机移动本身就会造成图像中的像素位移。因此，图像运动不仅包含物体运动，也包含由相机运动产生的视差。

#### 6.2 相机投影模型

世界坐标系中的三维点 $\mathbf{P}_w$ 在第 $i$ 个相机中的投影为：

$$
\tilde{\mathbf{p}}_i
\sim
K_i[R_i\mid\mathbf{t}_i]\tilde{\mathbf{P}}_w
$$

其中：

- $K_i$：第 $i$ 个相机的内参。
- $R_i,\mathbf{t}_i$：相机外参。
- $\mathbf{P}_w$：三维点。
- $\mathbf{p}_i$：图像中的对应点。

多个视角提供的投影约束可以共同帮助估计场景几何与相机运动。

#### 6.3 FlowMap 的概念

FlowMap 利用图像对应关系、光流和可微分几何来联合估计与场景重建相关的变量，例如：

- 相机姿态；
- 每帧深度；
- 相机内参（具体取决于方法的设置和输入条件）。

它将图像之间的运动线索与三维投影关系联系起来，而不是把光流仅仅作为最终输出。

具体实现可能假设相机内参已知，也可能尝试估计内参。相机姿态、深度和内参能否同时可靠恢复，取决于观测数据、运动范围、场景假设及优化约束。

#### 6.4 单目重建中的尺度歧义

仅从单目图像恢复三维结构时，经常存在绝对尺度歧义。

如果把场景中所有三维点和相机平移同时乘以一个正比例因子，投影结果可能保持不变。

因此，重建结果不一定天然具有真实世界中的米制尺度。

这也是为什么多视图几何不仅需要对应点，还需要合理的相机运动、投影模型和必要的尺度约束。

#### 6.5 本节的整体联系

可以将这部分内容概括为：

1. Optical Flow 提供图像间的二维运动线索。
2. Point Tracking 维护跨多帧的点对应关系。
3. Scene Flow 关注三维场景点的运动。
4. FlowMap 等方法进一步结合相机模型与几何约束，恢复场景深度和相机运动。

这些方法关注的目标不同，但都与跨帧对应关系密切相关。

---

### 7. SIFT：尺度不变特征变换

#### 7.1 为什么需要特征匹配？

当两张图像来自不同的时间或视角时，直接比较像素往往不够可靠。

原因包括：

- 相机发生平移或旋转；
- 物体在图像中变大或变小；
- 光照和对比度变化；
- 图像存在局部遮挡；
- 相机视角改变。

SIFT（Scale-Invariant Feature Transform）是一种经典局部特征方法，主要用于检测稳定的关键点，并为关键点构造描述子，以便在不同图像中寻找对应关系。

SIFT 的经典流程包括：

1. 构建尺度空间。
2. 利用 Difference of Gaussians 检测关键点候选。
3. 筛除不稳定的候选点。
4. 为关键点分配主方向。
5. 构建局部特征描述子。
6. 匹配描述子。
7. 使用几何约束进一步筛除错误匹配。

#### 7.2 Scale Space（尺度空间）

图像中的同一个物体，在不同相机距离或图像分辨率下可能具有不同的大小。

因此，特征检测不能只在单一尺度下进行。

SIFT 通过对图像进行不同程度的高斯平滑，构建尺度空间：

$$
L(x,y,\sigma)=G(x,y,\sigma)*I(x,y)
$$

其中：

- $I(x,y)$：输入图像。
- $G(x,y,\sigma)$：标准差为 $\sigma$ 的二维高斯核。
- $*$：卷积。
- $\sigma$：尺度参数。

二维高斯核为：

$$
G(x,y,\sigma)
=
\frac{1}{2\pi\sigma^2}
\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)
$$

随着 $\sigma$ 增大，图像受到更强的平滑作用，细小结构逐渐消失。

在多个尺度上检测特征，可以让算法找到不同大小的稳定结构。

#### 7.3 Difference of Gaussians（DoG）

SIFT 使用高斯差分（DoG）来检测尺度空间中的关键点候选：

$$
D(x,y,\sigma)
=
L(x,y,k\sigma)-L(x,y,\sigma)
$$

其中 $k$ 是相邻尺度之间的比例因子。

DoG 可以近似尺度归一化的 Laplacian of Gaussian（LoG），差别在于近似关系中还存在与尺度相关的比例因子。

为什么使用 DoG？

- 高斯卷积可以利用已有的尺度空间结果高效计算。
- DoG 能突出不同尺度下的局部结构变化。
- 可以通过尺度空间中的极值寻找关键点候选。

因此，DoG 是 SIFT 检测关键点的核心工具之一。

#### 7.4 在三维尺度空间中寻找极值

SIFT 不只在图像平面中寻找极值，还在尺度维度上寻找极值。

对每个候选位置，比较：

- 当前尺度的 8 个相邻像素；
- 前一个尺度的 9 个位置；
- 后一个尺度的 9 个位置。

合计：

$$
8+9+9=26
$$

如果该位置的 DoG 值比周围 26 个邻居都大，或者都小，就可能是一个三维尺度空间中的局部极值。

这一步可以找到潜在的关键点候选。

#### 7.5 为什么还要筛选关键点？

局部极值并不意味着关键点一定稳定。

SIFT 主要处理两类不稳定候选：

**低对比度点：**

如果 DoG 响应很小，关键点可能容易受到噪声影响，因此需要剔除。

**边缘响应：**

某些位置沿着一个方向变化明显，但在另一个方向缺乏稳定的局部定位能力。

SIFT 会利用局部二阶结构筛除不稳定的边缘响应。

通过这些筛选，可以提高关键点定位的稳定性。

#### 7.6 Orientation Assignment（主方向分配）

为了减少图像旋转对特征匹配的影响，SIFT 为关键点分配一个主方向。

首先计算局部图像梯度：

$$
I_x=\frac{\partial L}{\partial x},
\qquad
I_y=\frac{\partial L}{\partial y}
$$

梯度幅值为：

$$
m(x,y)=\sqrt{I_x^2+I_y^2}
$$

梯度方向为：

$$
\theta(x,y)=\operatorname{atan2}(I_y,I_x)
$$

然后，在关键点附近收集梯度方向信息，并使用高斯权重强调关键点附近的梯度。

将这些方向统计到方向直方图中，选择主要峰值作为关键点的主方向。

如果局部区域中存在多个明显方向峰值，也可能为同一个位置生成多个方向不同的关键点。

#### 7.7 为什么分配主方向能够实现旋转不变性？

假设一张图像整体旋转了角度 $\alpha$，那么局部梯度方向也会相应旋转。

如果描述子直接使用原始梯度方向，旋转前后的描述子就可能出现较大差异。

SIFT 将局部方向参考系对齐到关键点的主方向，相当于消除整体旋转带来的方向偏移。

这样，同一个局部结构在不同旋转角度下，就更有机会生成相似的描述子。

#### 7.8 SIFT Descriptor（局部特征描述子）

经典 SIFT 描述子是一个 128 维向量。

它的构造过程大致为：

1. 以关键点为中心，选取局部图像区域。
2. 将区域划分成 $4\times4$ 个空间单元。
3. 每个单元统计 8 个方向区间的梯度直方图。
4. 拼接所有空间单元的直方图。

因此，描述子维度为：

$$
4\times4\times8=128
$$

其中：

- $4\times4$：描述局部区域的空间布局。
- $8$：每个空间单元的方向统计维度。
- $128$：最终描述子长度。

它不仅记录有哪些梯度方向，还保留了这些方向在局部空间中的分布情况。

为了减弱亮度和对比度变化的影响，SIFT 还会对描述子进行归一化等处理。

#### 7.9 Descriptor Matching（描述子匹配）

得到两张图像的 SIFT 描述子后，可以计算它们之间的距离。

常见做法是使用欧氏距离：

$$
d(\mathbf{f}_1,\mathbf{f}_2)
=
\|\mathbf{f}_1-\mathbf{f}_2\|_2
$$

距离越小，意味着两个描述子越相似。

但只选择最近邻并不一定可靠，因为某个描述子可能与多个候选点都比较相似。

因此，SIFT 常结合 Lowe's Ratio Test（比率检验）。

设：

- $d_1$：最近邻描述子距离。
- $d_2$：次近邻描述子距离。

比率为：

$$
r=\frac{d_1}{d_2}
$$

当：

$$
r<\tau
$$

时，保留该匹配，其中 $\tau$ 是阈值，经典实践中常见取值约为 $0.7$，但具体阈值需要根据任务调整。

直观上：

- 最近邻明显比次近邻更相似：匹配相对明确。
- 最近邻和次近邻距离接近：匹配具有歧义。

比率检验用于剔除部分歧义匹配，但不能保证剩下的匹配全部正确。

#### 7.10 RANSAC：利用几何一致性剔除错误匹配

即使通过比率检验，仍可能存在错误对应点。

RANSAC（Random Sample Consensus）通过反复抽取少量匹配点，估计几何模型，并统计其他匹配点中有多少符合该模型。

典型流程：

1. 随机选取最小样本集。
2. 根据样本估计几何模型。
3. 检查所有匹配点是否符合模型。
4. 统计内点数量。
5. 重复多次，选择支持度较高的模型。
6. 使用内点进一步估计模型。

根据场景，可以选择不同几何模型：

- **单应矩阵（Homography）**：适用于平面场景、纯相机旋转等特定情况。
- **基础矩阵（Fundamental Matrix）**：用于一般双目视图中的极几何约束，不要求场景全部位于同一平面。

因此，不能把单应矩阵当作所有三维场景的通用几何模型。

#### 7.11 SIFT 的 OpenCV 实例

下面是一个简化的 SIFT 匹配流程：

```python
import cv2
import numpy as np

img1 = cv2.imread("image1.jpg", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("image2.jpg", cv2.IMREAD_GRAYSCALE)

# 1. 创建 SIFT 检测器并提取关键点与描述子
sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# 2. 使用 KNN 查找每个描述子的两个最近邻
matcher = cv2.BFMatcher()
pairs = matcher.knnMatch(des1, des2, k=2)

# 3. Lowe 比率检验
good = [
    m for m, n in pairs
    if m.distance < 0.7 * n.distance
]

print("通过比率检验的匹配数：", len(good))

# 4. 使用 RANSAC 估计单应矩阵
# 注意：该步骤适用于平面场景、纯旋转等符合单应模型的情况
if len(good) >= 4:
    pts1 = np.float32(
        [kp1[m.queryIdx].pt for m in good]
    ).reshape(-1, 1, 2)

    pts2 = np.float32(
        [kp2[m.trainIdx].pt for m in good]
    ).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(
        pts1, pts2, cv2.RANSAC, 3.0
    )

    if H is not None and mask is not None:
        print("RANSAC 内点数：", int(mask.sum()))
```

其中：

- `detectAndCompute`：检测关键点并计算描述子。
- `knnMatch(..., k=2)`：为每个描述子寻找两个最近邻。
- `m.distance`：最近邻的描述子距离。
- `n.distance`：次近邻的描述子距离。
- `findHomography`：使用 RANSAC 估计单应矩阵。
- `mask`：标记匹配点是否被判定为内点。

实际使用时，还应检查输入图像是否成功读取、描述子是否存在，以及匹配点数量是否足以支持几何估计。

---

### 8. 总结

#### 8.1 不同方法解决的问题

| 方法 | 输入与关注对象 | 核心输出 |
|---|---|---|
| Lucas–Kanade | 两帧图像中的局部窗口 | 局部光流 |
| RAFT | 两帧图像及其特征相关性 | 稠密二维光流 |
| Point Tracking | 查询点与视频帧序列 | 跨帧点轨迹 |
| TAPIR | 查询点与视频序列 | 长期点轨迹及可见性信息 |
| Scene Flow | 多帧场景及相关几何信息 | 三维场景运动 |
| FlowMap | 图像对应关系与相机几何 | 场景深度、相机运动等几何变量 |
| SIFT | 单张或多张图像 | 稳定关键点与局部描述子 |

#### 8.2 从二维对应关系到三维几何


1. **检测和描述局部特征**：SIFT 为图像中的关键点构建描述子。
2. **建立图像对应关系**：描述子匹配、光流或学习式匹配方法寻找对应点。
3. **维护跨帧轨迹**：Point Tracking、TAPIR 关注点在视频中的长期身份与位置。
4. **估计图像运动**：Optical Flow 描述两帧之间的稠密二维位移。
5. **引入三维信息**：Scene Flow 描述三维场景点的运动。
6. **利用相机几何**：FlowMap 等方法结合对应关系与投影模型恢复场景结构和相机运动。

要特别区分：**二维对应关系不等于三维重建，二维光流也不等于三维场景流。** 从二维图像恢复三维信息，需要进一步引入相机模型、多个视角或其他几何约束。





### Multi-View Geometry（多视图几何）

### 1. 多视图几何概述

#### 1.1 什么是多视图几何？

多视图几何（Multi-View Geometry）研究如何利用同一个场景在不同相机位置、不同视角或不同时间拍摄的图像，恢复相机运动与场景的三维结构。

单张图像将三维世界投影到二维平面，深度信息通常无法直接确定。

如果从多个视角观察同一个三维点，不同图像中的对应点会受到共同的三维几何关系约束。利用这些约束，可以估计：

- 相机的位置与朝向；
- 场景中三维点的位置；
- 场景的深度结构；
- 不同图像之间的几何关系。

多视图几何是 Structure from Motion（SfM）、双目立体视觉、三维重建、视觉定位和 SLAM 等任务的重要基础。

#### 1.2 从二维图像到三维结构的基本流程

典型的多视图几何流程可以概括为：

1. **特征检测与匹配**：利用 SIFT 或学习式特征匹配方法，寻找不同图像中的对应点。
2. **估计相机几何关系**：利用对应点估计基础矩阵、相对相机姿态等。
3. **恢复相机运动**：估计相机之间的相对旋转和平移方向。
4. **三角化**：利用多个视角的观测估计三维点。
5. **联合优化**：通过 Bundle Adjustment 优化相机参数和三维点位置。
6. **扩展重建结果**：将更多图像、特征点或深度信息加入重建过程。

实际系统中的执行顺序可能有所不同，部分步骤也可能联合进行。

#### 1.3 与上一节内容的联系

上一节学习的内容为多视图几何提供了对应关系：

- **SIFT**：检测局部关键点并构建描述子。
- **Optical Flow**：估计两帧图像之间的二维运动。
- **Point Tracking**：维护点在多帧视频中的轨迹。
- **RAFT 等学习式方法**：利用学习到的特征和相关性估计图像运动。

这些方法可以帮助找到图像中的对应点，但对应点本身并不直接等于三维结构。

要从二维对应关系恢复三维信息，还需要相机模型和几何约束。

---

### 2. 针孔相机模型（Pinhole Camera Model）

#### 2.1 基本原理

针孔相机模型将相机近似为一个投影中心。空间中的三维点、投影中心和对应的图像点位于同一条直线上。

假设三维点在相机坐标系中的坐标为：

$$
\mathbf{P}_c=
\begin{bmatrix}
X_c\\Y_c\\Z_c
\end{bmatrix}
$$

在理想针孔模型下，图像平面上的坐标为：

$$
x=f\frac{X_c}{Z_c}
$$

$$
y=f\frac{Y_c}{Z_c}
$$

其中：

- $f$：焦距。
- $X_c,Y_c,Z_c$：三维点在相机坐标系中的坐标。
- $x,y$：理想图像平面坐标。

这里假设相机坐标系的原点位于投影中心，$Z_c$ 轴沿相机光轴方向。

#### 2.2 为什么单张图像无法直接确定深度？

根据针孔相机模型：

$$
x=f\frac{X_c}{Z_c},\qquad
y=f\frac{Y_c}{Z_c}
$$

假设一个三维点为：

$$
\mathbf{P}_1=(1,0,10)^\top
$$

另一个三维点为：

$$
\mathbf{P}_2=(2,0,20)^\top
$$

取焦距 $f=800$，两者的水平投影分别为：

$$
x_1=800\frac{1}{10}=80
$$

$$
x_2=800\frac{2}{20}=80
$$

两点的竖直投影也相同，因此它们在图像中投影到同一个位置。

这说明单个像素通常只确定一条空间射线，无法单独确定该点沿射线方向的深度。

#### 2.3 齐次坐标与投影矩阵

为了将相机投影关系写成矩阵形式，引入齐次坐标：

$$
\tilde{\mathbf{P}}_c=
\begin{bmatrix}
X_c\\Y_c\\Z_c\\1
\end{bmatrix}
$$

图像点的齐次坐标为：

$$
\tilde{\mathbf{p}}=
\begin{bmatrix}
u\\v\\1
\end{bmatrix}
$$

在相机坐标系下，投影关系可以写为：

$$
\tilde{\mathbf{p}}\sim K[I\mid\mathbf{0}]\tilde{\mathbf{P}}_c
$$

其中 $K$ 为相机内参矩阵，$\sim$ 表示两侧齐次坐标相差一个非零比例因子。

齐次坐标的优势是能够将透视投影与坐标变换统一到矩阵表达中。

---

### 3. 相机内参（Camera Intrinsics）

#### 3.1 内参矩阵

真实图像使用像素坐标，而不是理想图像平面坐标。

相机内参矩阵通常表示为：

$$
K=
\begin{bmatrix}
f_x&s&c_x\\
0&f_y&c_y\\
0&0&1
\end{bmatrix}
$$

其中：

- $f_x,f_y$：水平方向和竖直方向的焦距参数，通常以像素为单位。
- $c_x,c_y$：主点坐标，通常接近图像中心。
- $s$：像素坐标轴之间的斜切参数，现代相机中通常近似取零。

当 $s=0$ 时，像素坐标满足：

$$
u=f_x\frac{X_c}{Z_c}+c_x
$$

$$
v=f_y\frac{Y_c}{Z_c}+c_y
$$

#### 3.2 内参的作用

内参描述相机自身的成像参数，包括焦距和主点位置等。

它回答的问题是：

**已知一个点在相机坐标系中的位置，相机应该把它投影到图像的哪个像素？**

内参一般通过相机标定获得，也可以在部分重建方法中与其他变量一起估计。

需要注意：内参不描述相机在世界中的位置或朝向。

---

### 4. 相机外参（Camera Extrinsics）

#### 4.1 世界坐标系与相机坐标系

在三维重建中，通常同时使用：

- **世界坐标系**：用于描述场景中三维点的位置。
- **相机坐标系**：以相机投影中心为原点描述三维点。

设一个三维点在世界坐标系中的坐标为：

$$
\mathbf{P}_w=
\begin{bmatrix}
X_w\\Y_w\\Z_w
\end{bmatrix}
$$

在相机坐标系中的坐标为：

$$
\mathbf{P}_c=
\begin{bmatrix}
X_c\\Y_c\\Z_c
\end{bmatrix}
$$

它们通过旋转矩阵 $R$ 和平移向量 $\mathbf{t}$ 联系：

$$
\boxed{\mathbf{P}_c=R\mathbf{P}_w+\mathbf{t}}
$$

其中：

- $R\in\mathbb{R}^{3\times3}$：旋转矩阵。
- $\mathbf{t}\in\mathbb{R}^{3}$：平移向量。

这里采用的是从世界坐标系到相机坐标系的变换约定。

#### 4.2 相机中心与平移向量的区别

假设相机中心在世界坐标系中的位置为 $\mathbf{C}$，则：

$$
\mathbf{P}_c=R(\mathbf{P}_w-\mathbf{C})
$$

展开可得：

$$
\mathbf{P}_c=R\mathbf{P}_w-R\mathbf{C}
$$

与外参形式对比：

$$
\mathbf{P}_c=R\mathbf{P}_w+\mathbf{t}
$$

因此：

$$
\boxed{\mathbf{t}=-R\mathbf{C}}
$$

所以，平移向量 $\mathbf{t}$ 不一定等于相机中心在世界坐标系中的位置。

这一点在多视图几何中非常重要，因为不同教材可能采用不同的变换方向约定。

#### 4.3 完整相机投影模型

结合内参与外参，可以得到完整投影公式：

$$
\boxed{
\tilde{\mathbf{p}}
\sim K[R\mid\mathbf{t}]\tilde{\mathbf{P}}_w
}
$$

其中：

- $\tilde{\mathbf{P}}_w$：世界坐标系中的三维点。
- $[R\mid\mathbf{t}]$：世界坐标到相机坐标的变换。
- $K$：相机内参矩阵。
- $\tilde{\mathbf{p}}$：图像中的齐次像素坐标。

定义投影矩阵：

$$
P=K[R\mid\mathbf{t}]
$$

则：

$$
\tilde{\mathbf{p}}\sim P\tilde{\mathbf{P}}_w
$$

需要注意，投影矩阵 $P$ 是一个 $3\times4$ 矩阵，而 $\mathbf{P}_w$ 表示三维点，两者是不同的符号对象。

---

### 5. 三角化（Triangulation）

#### 5.1 三角化的基本思想

三角化是利用同一个三维点在多个视角中的二维观测，估计其三维位置的过程。

假设两台相机观察同一个场景点，并且已知：

- 两台相机的内参；
- 两台相机的相对姿态；
- 两张图像中的对应点。

每个像素点都对应一条从相机中心出发的空间射线。

如果两条射线相交，交点就是理想情况下的三维点。

真实数据存在测量误差时，两条射线通常不会严格相交，因此需要通过数值方法估计最合理的三维位置。

#### 5.2 从像素点反投影到空间射线

已知像素点的齐次坐标：

$$
\tilde{\mathbf{p}}=
\begin{bmatrix}
u\\v\\1
\end{bmatrix}
$$

利用内参矩阵的逆，可以得到相机坐标系中的射线方向：

$$
\mathbf{d}_c\propto K^{-1}\tilde{\mathbf{p}}
$$

其中 $\mathbf{d}_c$ 只确定方向，不确定距离。

如果相机中心在世界坐标系中的位置为 $\mathbf{C}$，相机到世界坐标系的旋转矩阵为 $R^\top$，那么射线方向转换到世界坐标系后为：

$$
\mathbf{d}_w=R^\top\mathbf{d}_c
$$

对应的空间射线可以写为：

$$
\boxed{
\mathbf{X}(\lambda)=\mathbf{C}+\lambda\mathbf{d}_w
}
$$

其中 $\lambda>0$ 表示沿射线方向的距离参数。射线方向是否归一化不影响其几何意义，但会影响参数 $\lambda$ 的具体数值含义。

#### 5.3 双视角三角化

对于两台相机，分别得到：

$$
\mathbf{X}_1(\lambda_1)
=
\mathbf{C}_1+\lambda_1\mathbf{d}_1
$$

$$
\mathbf{X}_2(\lambda_2)
=
\mathbf{C}_2+\lambda_2\mathbf{d}_2
$$

在理想情况下：

$$
\mathbf{C}_1+\lambda_1\mathbf{d}_1
=
\mathbf{C}_2+\lambda_2\mathbf{d}_2
$$

通过求解 $\lambda_1,\lambda_2$，可以确定三维点。

在有噪声的情况下，可以寻找使两条射线之间距离最小的点，或者直接最小化各图像中的重投影误差。

#### 5.4 重投影误差（Reprojection Error）

得到三维点估计 $\hat{\mathbf{P}}$ 后，可以将它重新投影到第 $i$ 张图像中：

$$
\hat{\mathbf{p}}_i
=
\pi\left(K_i(R_i\hat{\mathbf{P}}+\mathbf{t}_i)\right)
$$

其中 $\pi$ 表示透视除法，将齐次坐标转换成普通像素坐标。

重投影误差定义为：

$$
e_i=\|\mathbf{p}_i-\hat{\mathbf{p}}_i\|_2
$$

其中：

- $\mathbf{p}_i$：图像中实际检测到的对应点。
- $\hat{\mathbf{p}}_i$：三维点重新投影得到的预测像素坐标。

重投影误差越小，说明估计的三维点与图像观测越一致。

不过，重投影误差较小并不保证三维点一定准确。例如，在视差很小的情况下，深度可能变化很大，而像素重投影变化仍然很小。

---

### 6. 视差与深度估计

#### 6.1 什么是视差？

视差（Parallax）指同一个三维点在不同相机视角下产生的图像位置差异。

对于经过校正的平行双目相机，左右图像中的对应点通常位于相同的图像行上。

设左图像中的水平坐标为 $u_L$，右图像中的水平坐标为 $u_R$，则视差定义为：

$$
d=u_L-u_R
$$

这里采用左图减右图的符号约定；不同系统可能采用不同约定。

#### 6.2 双目深度公式

对于焦距为 $f$、基线距离为 $B$ 的理想平行双目系统：

$$
\boxed{Z=\frac{fB}{d}}
$$

其中：

- $Z$：目标的深度。
- $f$：焦距，通常以像素为单位。
- $B$：两相机中心之间的距离。
- $d$：视差，单位为像素。

如果 $f$ 以像素为单位、$B$ 以米为单位，那么 $Z$ 的单位为米。

例如：

$$
f=800,\qquad B=0.2\text{ m},\qquad d=40
$$

则：

$$
Z=\frac{800\times0.2}{40}=4\text{ m}
$$

#### 6.3 为什么远处物体的深度更难估计？

根据双目深度公式：

$$
Z=\frac{fB}{d}
$$

深度与视差成反比。

对视差求导：

$$
\frac{\partial Z}{\partial d}
=
-\frac{fB}{d^2}
$$

因此，当视差较小时，视差测量误差会引起更大的深度误差。

这意味着：

- 近处物体通常具有较大视差。
- 远处物体通常具有较小视差。
- 视差越小，深度估计往往越敏感。

增加相机基线通常可以增大视差，但基线过大也可能导致图像重叠区域减小、遮挡增多和匹配难度增加。

#### 6.4 三角化的主要误差来源

- 图像对应点定位不准确。
- 相机内参标定存在误差。
- 相机相对姿态估计不准确。
- 两个视角之间的基线太小。
- 目标距离太远，视差过小。
- 场景发生运动，违背静态场景假设。

因此，三角化的质量不仅取决于算法，还取决于图像匹配质量和相机几何条件。

---

### 7. 极几何（Epipolar Geometry）

#### 7.1 为什么需要极几何？

假设我们已经知道第一张图像中的一个点，并希望在第二张图像中找到它的对应点。

如果不使用任何几何信息，就可能需要在第二张图像的大范围区域中搜索。

但是，两台相机观察同一个三维点时，两个相机中心与该三维点共同确定一个平面，称为极平面（Epipolar Plane）。

这个平面与两张图像平面的交线分别构成极线（Epipolar Line）。

因此，第一张图像中的一个点，会约束其对应点在第二张图像中的位置：对应点必须位于一条特定的极线上。

这可以显著缩小匹配搜索范围。

#### 7.2 极点与极线

- **极点（Epipole）**：一个相机中心在另一个相机图像平面上的投影。
- **极平面（Epipolar Plane）**：由两个相机中心和空间点共同确定的平面。
- **极线（Epipolar Line）**：极平面与图像平面的交线。

对第一张图像中的一个点，另一张图像中的对应点必须满足相应的极线约束。

#### 7.3 极线约束

设第一张图像中的齐次像素坐标为 $\mathbf{x}$，第二张图像中的对应点为 $\mathbf{x}'$。

它们满足：

$$
\boxed{
\mathbf{x}'^\top F\mathbf{x}=0
}
$$

其中 $F$ 是基础矩阵（Fundamental Matrix）。

给定第一张图像中的点 $\mathbf{x}$，第二张图像中的对应极线为：

$$
\mathbf{l}'=F\mathbf{x}
$$

因为直线可以表示为：

$$
\mathbf{l}'=
\begin{bmatrix}
a\\b\\c
\end{bmatrix}
$$

那么该极线上的任意点 $\mathbf{x}'=(u',v',1)^\top$ 都满足：

$$
au'+bv'+c=0
$$

这意味着对应点的搜索范围从整张图像缩小到了极线附近。

实际算法还需要考虑定位误差，因此通常会在极线附近的一定容差范围内搜索。

---

### 8. 基础矩阵（Fundamental Matrix）

#### 8.1 基础矩阵的作用

基础矩阵 $F$ 描述两张未必经过校正的图像之间的极几何关系。

它满足：

$$
\mathbf{x}'^\top F\mathbf{x}=0
$$

基础矩阵具有以下性质：

- $F$ 是一个 $3\times3$ 矩阵。
- 对于一般双目几何，$F$ 的秩为 2。
- $F$ 只确定到一个非零比例因子。
- 它不要求场景中的所有点都位于同一平面。

与单应矩阵不同，基础矩阵描述的是一般双视图的极几何关系，而不是将整张图像中的点统一映射到另一张图像。

#### 8.2 基础矩阵与相机参数的关系

设第一台相机为参考坐标系，第二台相机相对于第一台相机的旋转和平移为 $R,\mathbf{t}$。

在常见的相机变换约定下，基础矩阵满足：

$$
\boxed{
F=K_2^{-\top}[\mathbf{t}]_\times R K_1^{-1}
}
$$

其中：

- $K_1,K_2$：两台相机的内参矩阵。
- $R,\mathbf{t}$：两台相机之间的相对旋转和平移。
- $[\mathbf{t}]_\times$：由平移向量构成的反对称矩阵。

若：

$$
\mathbf{t}=
\begin{bmatrix}
t_x\\t_y\\t_z
\end{bmatrix}
$$

则：

$$
[\mathbf{t}]_\times=
\begin{bmatrix}
0&-t_z&t_y\\
t_z&0&-t_x\\
-t_y&t_x&0
\end{bmatrix}
$$

该矩阵满足：

$$
[\mathbf{t}]_\times\mathbf{a}
=
\mathbf{t}\times\mathbf{a}
$$

因此，基础矩阵将相机相对运动、内参与图像中的极几何关系联系起来。

注意：如果相机坐标变换的方向约定不同，相关公式的写法也可能不同，但几何含义是一致的。

---

### 9. 本质矩阵（Essential Matrix）

#### 9.1 本质矩阵与基础矩阵的区别

基础矩阵直接作用于像素齐次坐标。

如果使用内参矩阵对图像点进行归一化，得到相机归一化坐标：

$$
\hat{\mathbf{x}}=K_1^{-1}\mathbf{x}
$$

$$
\hat{\mathbf{x}}'=K_2^{-1}\mathbf{x}'
$$

那么可以用本质矩阵 $E$ 表示极几何约束：

$$
\boxed{
\hat{\mathbf{x}}'^\top E\hat{\mathbf{x}}=0
}
$$

本质矩阵与基础矩阵的关系为：

$$
\boxed{
E=K_2^\top F K_1
}
$$

在相对位姿约定下：

$$
\boxed{
E=[\mathbf{t}]_\times R
}
$$

因此：

- **基础矩阵 $F$**：作用于像素坐标，包含相机内参的影响。
- **本质矩阵 $E$**：作用于归一化相机坐标，描述相对旋转和平移方向所决定的极几何关系。

#### 9.2 从本质矩阵恢复相对相机姿态

当相机内参已知时，可以从对应点估计本质矩阵。

随后通过对本质矩阵进行奇异值分解，可以恢复相对旋转 $R$ 和平移方向。

但是，这个过程存在多个候选解，需要利用三维点应位于相机前方等条件进行筛选。

另外，单目双视图几何通常只能恢复平移方向，无法仅凭图像对应关系确定平移的绝对尺度。

---

### 10. Eight-Point Algorithm（八点算法）

#### 10.1 基本思想

八点算法用于根据两张图像中的对应点估计基础矩阵。

每一组对应点满足：

$$
\mathbf{x}'^\top F\mathbf{x}=0
$$

设：

$$
\mathbf{x}=
\begin{bmatrix}
u\\v\\1
\end{bmatrix},
\qquad
\mathbf{x}'=
\begin{bmatrix}
u'\\v'\\1
\end{bmatrix}
$$

将基础矩阵展开：

$$
F=
\begin{bmatrix}
f_{11}&f_{12}&f_{13}\\
f_{21}&f_{22}&f_{23}\\
f_{31}&f_{32}&f_{33}
\end{bmatrix}
$$

代入极线约束，可得：

$$
u'u f_{11}
+u'v f_{12}
+u'f_{13}
+v'u f_{21}
+v'v f_{22}
+v'f_{23}
+u f_{31}
+v f_{32}
+f_{33}
=0
$$

每一组对应点都提供一个关于基础矩阵元素的线性约束。

#### 10.2 转换成线性方程组

将基础矩阵元素排列成向量：

$$
\mathbf{f}=
\begin{bmatrix}
f_{11}&f_{12}&f_{13}&
f_{21}&f_{22}&f_{23}&
f_{31}&f_{32}&f_{33}
\end{bmatrix}^{\top}
$$

每一组对应点可以产生一行系数：

$$
\begin{bmatrix}
u'u&u'v&u'&v'u&v'v&v'&u&v&1
\end{bmatrix}
$$

收集多组对应点后，得到：

$$
A\mathbf{f}=0
$$

至少需要八组一般位置的对应点，才能在忽略基础矩阵秩约束之前获得非平凡的线性解。

#### 10.3 求解与秩约束

使用奇异值分解（SVD）求解：

$$
A=U\Sigma V^\top
$$

取 $V$ 中对应最小奇异值的右奇异向量作为初始解，并将其重新排列为 $3\times3$ 矩阵。

但是，线性求解得到的矩阵不一定严格满足秩为 2 的要求。

因此，对初始估计进行 SVD：

$$
F=U_F\Sigma_FV_F^\top
$$

设：

$$
\Sigma_F=
\begin{bmatrix}
\sigma_1&0&0\\
0&\sigma_2&0\\
0&0&\sigma_3
\end{bmatrix}
$$

将最小奇异值设为零：

$$
\Sigma_F'=
\begin{bmatrix}
\sigma_1&0&0\\
0&\sigma_2&0\\
0&0&0
\end{bmatrix}
$$

最终得到满足秩 2 约束的估计：

$$
F'=U_F\Sigma_F'V_F^\top
$$

实际使用中，通常还会进行坐标归一化，并结合 RANSAC 等方法处理错误匹配。

#### 10.4 为什么八点算法需要鲁棒估计？

特征匹配中可能包含错误对应点。

如果直接使用全部匹配点求解基础矩阵，少数错误匹配就可能严重影响估计结果。

因此，常见做法是：

1. 使用 RANSAC 随机抽取对应点子集。
2. 用子集估计候选基础矩阵。
3. 统计满足极线约束的内点。
4. 重复采样，选取支持度较高的模型。
5. 用最终内点重新估计基础矩阵。

八点算法提供的是基础矩阵的估计方法，RANSAC 则用于提高面对错误匹配时的鲁棒性。

---

### 11. Homography（单应矩阵）

#### 11.1 单应矩阵的定义

单应矩阵 $H$ 描述两个图像平面之间的投影变换：

$$
\boxed{
\tilde{\mathbf{x}}'\sim H\tilde{\mathbf{x}}
}
$$

其中：

- $\tilde{\mathbf{x}}$：第一张图像中的齐次像素坐标。
- $\tilde{\mathbf{x}}'$：第二张图像中的齐次像素坐标。
- $H$：$3\times3$ 单应矩阵。

单应矩阵有 9 个元素，但整体尺度不影响投影，因此具有 8 个自由度。

一般需要至少四组具有合适几何分布的点对应关系，才能估计单应矩阵。

#### 11.2 单应矩阵适用的情况

典型适用情况包括：

- 场景点全部位于同一平面。
- 相机只发生旋转，没有相机中心的平移。
- 某些可以用平面投影关系近似描述的场景。

对于一般具有不同深度的三维场景，如果相机中心发生平移，通常不能用一个单应矩阵准确描述所有点的对应关系。

#### 11.3 Homography 与 Fundamental Matrix 的区别

| 比较维度 | Homography $H$ | Fundamental Matrix $F$ |
|---|---|---|
| 主要描述 | 图像间的平面投影变换 | 两视图之间的极几何关系 |
| 典型约束 | $\mathbf{x}'\sim H\mathbf{x}$ | $\mathbf{x}'^\top F\mathbf{x}=0$ |
| 矩阵形状 | $3\times3$ | $3\times3$ |
| 典型用途 | 平面映射、图像拼接、纯旋转情况 | 一般双视图几何、极线搜索 |
| 场景限制 | 需要平面或特定相机运动条件 | 不要求场景全部位于同一平面 |

两者都可以用 RANSAC 估计，但它们表达的是不同的几何关系，不能随意互换。

---

### 12. Bundle Adjustment（光束法平差）

#### 12.1 为什么需要 Bundle Adjustment？

前面的步骤可以分别估计相机姿态和三维点，但这些估计往往存在误差。

Bundle Adjustment（BA）通过同时优化相机参数和三维点位置，使它们能够更好地解释所有图像观测。

其核心目标是最小化重投影误差。

#### 12.2 优化目标

假设：

- $\mathbf{P}_j$：第 $j$ 个三维点。
- $\mathbf{p}_{ij}$：第 $i$ 张图像中观测到的第 $j$ 个点。
- $\pi_i(\cdot)$：第 $i$ 个相机的投影函数。
- $\theta_i$：第 $i$ 个相机的参数，包括外参以及可能参与优化的内参。

则 BA 的目标可以写成：

$$
\boxed{
\min_{\{\theta_i\},\{\mathbf{P}_j\}}
\sum_{i,j}
\left\|
\mathbf{p}_{ij}
-
\pi_i(\mathbf{P}_j;\theta_i)
\right\|_2^2
}
$$

其中，求和只针对实际观测到的点与相机组合。

#### 12.3 优化的变量

根据任务设置，BA 可以优化：

- 相机旋转和平移；
- 三维点坐标；
- 相机内参；
- 其他与相机模型相关的参数。

如果内参已通过标定获得并且保持固定，那么优化变量可以只包含相机外参与三维点。

#### 12.4 BA 的特点

BA 通常涉及大量变量，但每个图像观测只依赖对应的相机和三维点，因此优化问题具有稀疏结构。

常见的优化方法包括 Gauss–Newton 和 Levenberg–Marquardt。

BA 能够改善整体几何一致性，但不能自动解决所有歧义。例如，单目重建中的整体尺度歧义仍然需要通过适当的约束或参考信息处理。

---

### 13. Structure from Motion（SfM）

#### 13.1 SfM 的定义

Structure from Motion（SfM，运动恢复结构）是指利用多张图像，联合估计相机运动与场景三维结构的过程。

输入通常是一组从不同视角拍摄的图像，输出通常包括：

- 相机姿态；
- 稀疏三维点云；
- 图像之间的对应关系；
- 相机参数及重建结果。

#### 13.2 典型 SfM 流程

1. **特征提取**：在每张图像中检测关键点并计算描述子。
2. **特征匹配**：建立不同图像之间的点对应关系。
3. **几何验证**：使用基础矩阵、单应矩阵或其他几何模型剔除错误匹配。
4. **初始化相机姿态**：从合适的图像对估计相对相机运动。
5. **三角化**：恢复初始三维点。
6. **逐步注册更多图像**：估计新相机姿态并加入更多三维点。
7. **Bundle Adjustment**：联合优化相机参数和三维点。
8. **过滤异常点**：根据重投影误差、可见性和几何一致性筛除不可靠的点。

不同 SfM 系统的具体流程可能不同，但核心目标都是建立图像观测、相机运动与三维结构之间的一致关系。

#### 13.3 SfM 的局限

- 纹理不足会使特征匹配困难。
- 重复纹理可能产生错误对应。
- 相机运动太小会导致三角化不稳定。
- 动态物体会破坏静态场景假设。
- 单目重建通常存在尺度歧义。
- 长序列可能产生累计误差，需要回环或全局优化等机制改善。

---

### 14. 多视图几何的核心概念对比

| 概念 | 主要解决的问题 |
|---|---|
| Pinhole Camera Model | 三维点如何投影到二维图像 |
| Intrinsics $K$ | 相机如何成像 |
| Extrinsics $R,\mathbf{t}$ | 世界坐标系与相机坐标系如何转换 |
| Triangulation | 如何从多个视角的二维观测估计三维点 |
| Parallax | 不同视角之间的观测差异如何提供深度线索 |
| Fundamental Matrix $F$ | 两视图之间的极几何约束 |
| Essential Matrix $E$ | 归一化相机坐标下的极几何约束 |
| Eight-Point Algorithm | 如何从对应点估计基础矩阵 |
| Homography $H$ | 如何描述特定条件下的图像平面投影变换 |
| Bundle Adjustment | 如何联合优化相机参数和三维点 |
| SfM | 如何从多张图像恢复相机运动与场景结构 |

---

### 15. 总结

#### 15.1 知识链

多视图几何的核心逻辑可以概括为：

1. 通过 SIFT、光流或学习式方法获得图像对应点。
2. 使用相机内参与外参描述成像过程。
3. 利用多视角几何关系建立极线约束。
4. 估计相机之间的相对姿态。
5. 通过三角化恢复三维点。
6. 使用 Bundle Adjustment 优化整体重建结果。
7. 通过 SfM 等流程逐步构建更完整的三维场景。

#### 15.2 公式

**针孔相机模型：**

$$
x=f\frac{X_c}{Z_c},
\qquad
y=f\frac{Y_c}{Z_c}
$$

**相机外参：**

$$
\mathbf{P}_c=R\mathbf{P}_w+\mathbf{t}
$$

**完整投影模型：**

$$
\tilde{\mathbf{p}}
\sim
K[R\mid\mathbf{t}]\tilde{\mathbf{P}}_w
$$

**双目深度公式：**

$$
Z=\frac{fB}{d}
$$

**极线约束：**

$$
\mathbf{x}'^\top F\mathbf{x}=0
$$

**本质矩阵：**

$$
E=[\mathbf{t}]_\times R
$$

**单应矩阵：**

$$
\tilde{\mathbf{x}}'\sim H\tilde{\mathbf{x}}
$$

**Bundle Adjustment：**

$$
\min_{\{\theta_i\},\{\mathbf{P}_j\}}
\sum_{i,j}
\left\|
\mathbf{p}_{ij}
-
\pi_i(\mathbf{P}_j;\theta_i)
\right\|_2^2
$$

#### 15.3 

- 内参描述相机成像参数，外参描述坐标系之间的变换。
- 相机中心 $\mathbf{C}$ 与外参平移向量 $\mathbf{t}$ 不一定相同。
- 单张图像通常不能确定三维点的深度。
- 三角化需要多个视角的对应关系以及相机几何信息。
- 视差越小，深度估计通常越敏感。
- 基础矩阵约束对应点位于极线上，但不直接给出三维点。
- 本质矩阵作用于归一化相机坐标，基础矩阵作用于像素齐次坐标。
- 单应矩阵不能作为一般三维场景中所有点对应关系的通用模型。
- 三角化与 Bundle Adjustment 不同：前者估计三维点，后者联合优化相机参数与三维点。
- 单目多视图重建通常存在整体尺度歧义。

注意**图像对应关系提供观测，相机模型提供几何约束，多视图几何则利用二者共同恢复相机运动与三维结构。**


### 1. Differentiable Rendering: Data Structures and Signal Parameterizations

### 1.1 概览

**如何用不同的数据结构和数学函数表示三维场景，以及这些表示方式如何影响渲染、优化和计算效率？**

学习 CNN、Transformer、ViT、扩散模型等方法时，通常将图像作为输入，通过神经网络得到分类结果、特征或生成图像。

在三维计算机视觉中，我们还需要考虑一个问题：如何建立能够描述三维世界的表示，并从不同相机视角生成对应的二维图像？

例如，给定多张从不同视角拍摄的照片，我们希望恢复场景的几何结构、颜色和外观。为了实现这一目标，需要先确定三维场景的表示方式，再利用图像监督优化场景参数。

现在主要关注以下问题：

1. 显式表示与隐式表示有什么区别？
2. 网格、体素、SDF 和神经场分别适合什么任务？
3. 为什么要将离散数据结构与神经网络结合？
4. 可微渲染如何将图像误差传递回三维场景参数？
5. NeRF 和 3D Gaussian Splatting 在表示方式上有什么区别？

---

### 2. 从三维场景到二维图像

#### 2.1 渲染的基本过程

假设我们已经拥有一个三维场景，场景参数记为 $\theta$，相机参数记为 $c$。

渲染器根据场景和相机生成图像：

$$
\hat{I}=\mathcal{R}(\theta,c)
$$

其中：

- $\theta$：场景参数，可以包括几何、颜色、密度、材质等；
- $c$：相机参数，包括相机位置、朝向和投影参数；
- $\mathcal{R}$：渲染函数；
- $\hat{I}$：生成的图像。

如果有真实图像 $I$，就可以比较生成图像与真实图像之间的差异。

例如，定义像素级平方误差：

$$
\mathcal{L}(\theta)
=
\frac{1}{N}
\sum_{i=1}^{N}
\left\|
\hat{I}_i-I_i
\right\|_2^2
$$

这里的 $N$ 表示像素数量，$\hat{I}_i$ 和 $I_i$ 分别表示第 $i$ 个像素的预测颜色与真实颜色。

我们的目标是找到合适的场景参数，使损失尽可能小：

$$
\theta^*
=
\arg\min_{\theta}
\mathcal{L}(\theta)
$$

这就是基于图像监督进行三维场景优化的基本形式。

#### 2.2 为什么要研究场景表示？

假设我们要描述一个房间中的桌子。

可以使用以下方式：

- 用三角网格描述桌面的顶点和边；
- 用体素网格记录哪些空间位置被桌子占据；
- 用 SDF 函数描述空间位置到桌面表面的距离；
- 用神经网络根据空间坐标预测桌子的几何与外观；
- 用多个三维高斯基元近似桌子的表面与外观。

这些方法都可能描述同一个物体，但它们在存储、查询、渲染和优化方面的特性不同。

因此，场景表示并非简单的存储格式，而是会直接影响整个系统的设计。

---

### 3. 显式表示与隐式表示

#### 3.1 显式表示（Explicit Representation）

显式表示直接存储构成场景的几何元素及其属性。

常见例子包括：

- 三角网格；
- 点云；
- 体素网格；
- 3D Gaussian Splatting 中的高斯基元。

例如，三维点可以表示为：

$$
\mathbf{p}_i=
\begin{bmatrix}
x_i\\
y_i\\
z_i
\end{bmatrix}
$$

一个点还可以附带颜色、法向量等属性。

显式表示的主要特点是，场景中的元素具有明确的存储结构，通常可以直接访问和操作。

#### 3.2 隐式表示（Implicit Representation）

隐式表示不一定显式存储完整的表面，而是使用一个函数描述空间属性。

例如：

$$
f:\mathbb{R}^3\rightarrow\mathbb{R}
$$

输入三维坐标，输出一个标量。

如果函数描述的是有符号距离，那么物体表面可以由函数的零等值面表示：

$$
\mathcal{S}
=
\left\{
\mathbf{x}\in\mathbb{R}^3
\mid f(\mathbf{x})=0
\right\}
$$

如果函数描述的是占据概率，则可以根据输出值判断某个位置是否属于物体。

隐式表示的关键在于：我们通过函数查询空间位置的属性，而不是必须为每个表面点单独存储几何信息。

#### 3.3 显式和隐式的区别

| 对比维度 | 显式表示 | 隐式表示 |
|---|---|---|
| 基本形式 | 存储几何元素 | 定义空间函数 |
| 查询方式 | 访问点、面或空间单元 | 将坐标输入函数 |
| 几何表达 | 通常直接存储几何 | 可通过等值面表达几何 |
| 细节控制 | 依赖基元密度和结构 | 依赖函数表达能力和采样方式 |
| 典型例子 | Mesh、Point Cloud、3D Gaussians | SDF、Occupancy Field、Neural Field |

需要注意，显式表示与隐式表示不是绝对对立的两类。一个系统可以使用显式网格存储特征，同时利用神经网络预测最终的场景属性。

---

### 4. 基于表面的表示：Mesh 与 Point Cloud

#### 4.1 三角网格（Triangle Mesh）

三角网格是计算机图形学中非常经典的表面表示方式。

一个网格可以表示为：

$$
\mathcal{M}=(V,F)
$$

其中：

- $V$ 是顶点集合；
- $F$ 是三角形面片集合。

顶点集合可以写为：

$$
V=\{\mathbf{v}_1,\mathbf{v}_2,\ldots,\mathbf{v}_N\}
$$

每个三角形由三个顶点的索引定义。

例如，一个三角形由三个三维顶点构成：

$$
\mathbf{v}_1=(x_1,y_1,z_1)
$$

$$
\mathbf{v}_2=(x_2,y_2,z_2)
$$

$$
\mathbf{v}_3=(x_3,y_3,z_3)
$$

三角形内部的点可以使用重心坐标表示：

$$
\mathbf{x}
=
\lambda_1\mathbf{v}_1+
\lambda_2\mathbf{v}_2+
\lambda_3\mathbf{v}_3
$$

其中：

$$
\lambda_1+\lambda_2+\lambda_3=1
$$

并且：

$$
\lambda_1,\lambda_2,\lambda_3\geq 0
$$

重心坐标可以用于插值顶点颜色、纹理坐标和其他属性。

**优点：**

- 表面结构明确；
- 适合传统光栅化渲染；
- 便于进行几何编辑；
- 对具有清晰表面的物体比较自然。

**局限：**

- 需要维护顶点与面之间的连接关系；
- 复杂拓扑的优化可能比较困难；
- 表面可见性变化会使渲染梯度处理变得复杂；
- 对体积、半透明物体或复杂外观的表达，往往需要额外的机制。

需要注意：网格顶点坐标可以是连续可优化的，但这并不意味着整个网格渲染过程天然处处可微。

#### 4.2 点云（Point Cloud）

点云使用一组三维点描述物体：

$$
P=\{\mathbf{p}_1,\mathbf{p}_2,\ldots,\mathbf{p}_N\}
$$

点云不一定保存点之间的连接关系，因此很适合表达从深度相机或多视图重建得到的几何采样。

不过，数学上的点没有面积。如果直接把它们投影到图像平面，可能产生空洞，难以形成连续的表面。

一种解决方式是为点分配一定的覆盖范围，或者将点扩展成圆盘、椭圆和其他可渲染基元。

3D Gaussian Splatting 采用的就是一种更丰富的基元表示：它不只存储点的位置，还存储具有空间范围的三维高斯及其外观属性。

---

### 5. 体素网格与体积表示

#### 5.1 什么是体素？

二维图像由像素组成，而三维规则网格可以由体素（Voxel）组成。

二维图像可以表示为：

$$
I[i,j]
$$

三维体素网格则可以表示为：

$$
V[i,j,k]
$$

每个体素可以存储占据状态、密度、颜色或其他特征。

最简单的二值占据网格为：

$$
V[i,j,k]\in\{0,1\}
$$

其中，1 表示该体素被物体占据，0 表示未占据。

实际系统也可以使用连续数值表达占据概率或空间密度。

#### 5.2 体素网格的内存复杂度

假设网格在三个维度上的分辨率均为 $N$，那么体素总数为：

$$
N_{\text{voxel}}=N^3
$$

如果每个体素存储 $C$ 个浮点特征，每个浮点数占 4 字节，则存储量约为：

$$
M=4N^3C\ \text{bytes}
$$

例如，分辨率从 $128^3$ 提高到 $256^3$，每个维度都扩大两倍，体素总数就会变为原来的八倍。

这就是三维规则网格面临的核心问题：**空间分辨率提高时，存储成本会按立方增长。**

#### 5.3 体素网格的优缺点

优点：

- 结构规则；
- 查询方便；
- 适合三维卷积；
- 容易与占据预测、体积重建等任务结合。

局限：

- 内存消耗大；
- 空白空间也可能占据存储；
- 固定分辨率难以同时满足大尺度场景和局部高精度细节的需求。

因此，研究者会使用稀疏网格、八叉树和其他自适应空间结构，以减少不必要的存储和计算。

---

### 6. 连续场：Occupancy Field、SDF 与 Density Field

#### 6.1 场（Field）的概念

场可以理解为一个定义在空间位置上的函数。

例如：

$$
f:\mathbb{R}^3\rightarrow\mathbb{R}
$$

对于任意三维位置 $\mathbf{x}$，函数都会返回一个数值。

根据函数的具体含义，可以得到不同的场景表示。

#### 6.2 占据场（Occupancy Field）

占据场描述空间位置是否属于物体。

可以使用概率形式：

$$
o(\mathbf{x})\in[0,1]
$$

其中，输出越接近 1，表示该位置越可能被物体占据；越接近 0，则表示越可能位于物体外部。

占据场可以用于三维形状重建，但它本身不一定直接提供到表面的距离信息。

#### 6.3 有符号距离场（Signed Distance Field, SDF）

SDF 描述空间位置到物体表面的有符号距离。

$$
s:\mathbb{R}^3\rightarrow\mathbb{R}
$$

例如，可以约定物体内部为负、表面为零、外部为正：

$$
s(\mathbf{x})<0
\quad \text{物体内部}
$$

$$
s(\mathbf{x})=0
\quad \text{物体表面}
$$

$$
s(\mathbf{x})>0
\quad \text{物体外部}
$$

物体表面就是 SDF 的零等值面：

$$
\mathcal{S}
=
\left\{
\mathbf{x}\mid s(\mathbf{x})=0
\right\}
$$

与占据场相比，SDF 不仅描述位置是否在物体内部，还提供了与表面距离有关的信息。

在理想的精确距离场中，离表面足够近且距离函数可微的位置，梯度方向与表面的法向方向有关。这使 SDF 在几何优化和表面法向估计中具有优势。

#### 6.4 密度场（Density Field）

密度场在体积渲染中十分重要。

$$
\sigma:\mathbb{R}^3\rightarrow\mathbb{R}_{\geq 0}
$$

它描述空间位置上的体密度。

与网格直接定义表面不同，密度场可以描述分布在三维空间中的体积信息。

例如，雾、烟雾和半透明物体都不一定适合用单一的封闭表面来描述。

在 NeRF 中，密度与颜色场共同用于沿相机射线生成图像。

---

### 7. 神经场（Neural Fields）

#### 7.1 基本思想

神经场使用神经网络表示定义在空间中的函数。

最一般的形式可以写为：

$$
f_\theta:\mathbb{R}^3\rightarrow\mathbb{R}^C
$$

其中：

- $\mathbf{x}\in\mathbb{R}^3$ 是三维空间坐标；
- $\theta$ 是网络参数；
- $C$ 是输出维度。

给定空间位置，神经网络就可以预测该位置对应的几何或外观属性。

例如：

$$
f_\theta(\mathbf{x})=s(\mathbf{x})
$$

表示网络预测 SDF。

也可以让网络预测颜色、密度或其他特征。

#### 7.2 为什么神经场可以表示连续空间？

假设我们使用一个 MLP 作为空间函数：

$$
f_\theta(\mathbf{x})
=
W_L\phi_{L-1}
\left(
W_{L-1}\cdots\phi_1(W_1\mathbf{x})
\right)
$$

其中，$W_l$ 表示网络权重，$\phi_l$ 表示激活函数。

这里的重点不是网络的具体层数，而是：输入是连续坐标，网络参数在整个空间范围内共享。

因此，不必为每一个三维位置单独存储一个值。我们可以在需要时将坐标输入网络，计算对应的场景属性。

不过，这不意味着神经网络能够无成本地查询任意位置。每次查询仍然需要进行计算，而且网络的表达能力与训练方式会影响最终效果。

#### 7.3 神经辐射场（NeRF）

NeRF 使用神经网络表示场景的体密度与方向相关颜色。

其典型形式为：

$$
f_\theta(\mathbf{x},\mathbf{d})
\rightarrow
\left(
\sigma(\mathbf{x}),
\mathbf{c}(\mathbf{x},\mathbf{d})
\right)
$$

其中：

- $\mathbf{x}$：三维空间位置；
- $\mathbf{d}$：观察方向；
- $\sigma(\mathbf{x})$：体密度；
- $\mathbf{c}(\mathbf{x},\mathbf{d})$：预测颜色。

为什么颜色还依赖观察方向？

因为真实物体可能存在镜面反射、高光等视角相关外观。同一空间位置从不同方向观察时，呈现的颜色不一定相同。

因此，NeRF 不只是一个三维 RGB 网格，而是一个将空间位置和观察方向映射到密度与颜色的函数。

#### 7.4 神经场的优缺点

优点：

- 能够表达连续的空间属性；
- 网络参数可以在空间范围内共享；
- 不受固定体素分辨率的直接限制；
- 可以通过梯度优化网络参数；
- 适合与多视图图像监督结合。

局限：

- 大量坐标查询可能导致较高的计算开销；
- 训练可能比较耗时；
- 高频细节不一定容易学习；
- 需要合理设计采样、位置编码和加速结构。

因此，神经场不是在所有方面都优于网格。它提供的是另一种函数参数化方式，而不是消除所有三维表示问题的万能方案。

---

### 8. 网格与自适应数据结构

神经场提供了连续的函数表示，但如果渲染一张图像需要查询大量空间位置，每次都运行完整的 MLP 可能十分昂贵。

因此，一个重要方向是：使用离散数据结构加速查询，同时保留连续的表达能力。

#### 8.1 网格插值

假设一维函数在规则网格位置保存了采样值：

$$
f_i=f(x_i)
$$

当查询位置位于相邻网格点之间时，可以使用线性插值：

$$
f(x)\approx(1-t)f_i+tf_{i+1}
$$

其中：

$$
t=\frac{x-x_i}{x_{i+1}-x_i}
$$

当查询位置位于网格单元内时，三维网格可以使用三线性插值（Trilinear Interpolation）。

设一个网格单元八个角点的特征为 $f_{abc}$，其中 $a,b,c\in\{0,1\}$，那么查询结果可以写为：

$$
f(\mathbf{x})
\approx
\sum_{a,b,c\in\{0,1\}}
w_{abc}f_{abc}
$$

其中，权重由查询点在单元内的相对位置决定，并满足：

$$
\sum_{a,b,c\in\{0,1\}}w_{abc}=1
$$

直观地说，离查询点越近的角点通常具有越大的插值权重。

这样就可以通过查表和插值获得特征，而不必为每个位置执行完整的神经网络。

#### 8.2 稀疏网格（Sparse Grid）

如果场景中只有少部分空间包含有效信息，那么存储所有网格单元就会造成浪费。

稀疏网格只存储需要保留的单元或特征，从而减少空白区域的内存开销。

代价是需要额外的数据结构来定位有效单元，因此访问方式可能比规则网格复杂。

#### 8.3 自适应数据结构（Adaptive Data Structures）

自适应数据结构根据空间复杂程度分配不同的分辨率。

例如：

- 几何变化平缓的区域采用较粗的网格；
- 表面附近使用更细的网格；
- 高频细节区域分配更多特征。

八叉树（Octree）是一种常见的层次化空间划分方式：一个三维空间单元可以递归划分为八个子单元。

其目的不是让整个空间都具有最高分辨率，而是把存储和计算资源集中在更需要精细表达的区域。

---

### 9. 混合表示：离散特征与神经网络

#### 9.1 为什么需要混合表示？

纯神经场具有连续、灵活的表达能力，但查询成本可能较高。

纯离散网格便于查表，却可能面临较大的内存开销和分辨率限制。

混合表示希望同时利用二者的优点。

一个通用形式为：

$$
f_\theta(\mathbf{x})
=
g_\theta
\left(
\mathbf{x},\mathbf{z}(\mathbf{x})
\right)
$$

其中：

- $\mathbf{z}(\mathbf{x})$：从网格或其他离散结构中查询得到的局部特征；
- $g_\theta$：神经网络解码器；
- $f_\theta(\mathbf{x})$：最终预测的场景属性。

整个过程可以分成两步：

1. 根据空间坐标查询局部特征；
2. 将坐标和特征输入神经网络，预测密度、颜色或几何属性。

这是一种重要的设计思路：离散结构负责高效存储和查询，神经网络负责学习复杂的特征映射。

#### 9.2 多分辨率特征

假设系统使用 $L$ 个不同分辨率的网格。

对空间位置 $\mathbf{x}$，分别查询各层特征，并进行拼接：

$$
\mathbf{z}(\mathbf{x})
=
\operatorname{Concat}
\left(
\mathbf{z}_1(\mathbf{x}),
\mathbf{z}_2(\mathbf{x}),
\ldots,
\mathbf{z}_L(\mathbf{x})
\right)
$$

其中，$\mathbf{z}_l(\mathbf{x})$ 表示第 $l$ 层网格在该位置提供的特征。

不同尺度通常承担不同的作用：

- 粗尺度特征有助于表示大范围结构；
- 细尺度特征有助于表示局部细节；
- 多尺度组合能够兼顾不同空间频率的信息。

随后，解码网络根据这些特征预测场景属性：

$$
f_\theta(\mathbf{x})
=
g_\theta\left(\mathbf{x},\mathbf{z}(\mathbf{x})\right)
$$

#### 9.3 哈希网格（Hash Grid）

一种代表性的混合表示思路是使用多分辨率哈希网格。

它将不同分辨率下的网格位置映射到有限大小的特征表，再通过插值得到查询特征。

这样可以减少直接存储完整高分辨率网格所需的内存。

不过，哈希映射可能导致不同空间位置映射到相同索引，产生特征冲突。因此，实际方法需要通过多尺度特征和优化过程共同处理这些问题。

这里需要记住的不是哈希表的具体实现，而是更普遍的思想：

**使用显式数据结构提高查询效率，再使用神经网络提高特征表达能力。**

---

### 10. 可微渲染（Differentiable Rendering）

前面的内容讨论了如何表示三维场景。现在考虑另一个问题：如何利用真实图像优化这些场景参数？

#### 10.1 从场景参数到图像损失

设场景参数为 $\theta$，相机参数为 $c$，渲染结果为：

$$
\hat I=\mathcal{R}(\theta,c)
$$

给定真实图像 $I$，定义平方误差损失：

$$
\mathcal{L}(\theta)
=
\frac{1}{N}
\sum_{i=1}^{N}
\left\|
\hat I_i-I_i
\right\|_2^2
$$

我们的目标是找到能够生成更接近真实图像的场景参数。

如果渲染器对参数可微，就可以通过链式法则计算梯度：

$$
\frac{\partial\mathcal{L}}{\partial\theta}
=
\frac{\partial\mathcal{L}}{\partial\hat I}
\frac{\partial\hat I}{\partial\theta}
$$

其中：

- $\frac{\partial\mathcal{L}}{\partial\hat I}$ 描述图像变化如何影响损失；
- $\frac{\partial\hat I}{\partial\theta}$ 描述场景参数变化如何影响渲染图像。

这与 CS231n 中的反向传播具有相同的基本思想。

#### 10.2 参数更新

得到梯度后，可以使用梯度下降更新场景参数：

$$
\theta_{t+1}
=
\theta_t
-
\eta
\frac{\partial\mathcal{L}}{\partial\theta}
$$

其中，$\eta$ 是学习率。

通过反复执行渲染、计算损失、反向传播和参数更新，场景表示可以逐步拟合观测图像。

#### 10.3 为什么可微渲染并不简单？

普通神经网络训练通常可以直接通过连续运算构建计算图，但渲染还涉及：

- 表面是否可见；
- 三角形或其他基元是否覆盖某个像素；
- 多个物体之间的遮挡关系；
- 射线与场景的相交；
- 离散采样和拓扑变化。

这些因素可能使渲染结果对某些参数的变化不连续，或者使梯度难以计算。

因此，**可微渲染不是简单地给传统渲染器套上自动求导，而是需要设计合适的渲染与梯度传播机制。**

不同方法会针对自身的表示方式，采用不同的近似、连续化或梯度估计策略。

---

### 11. NeRF 与 3D Gaussian Splatting

NeRF 和 3D Gaussian Splatting 都可以用于三维场景重建和新视角合成，但它们采用了不同的场景表示。

#### 11.1 NeRF：通过连续神经场表示场景

NeRF 使用神经网络预测空间位置的密度和方向相关颜色。

$$
f_\theta(\mathbf{x},\mathbf{d})
\rightarrow
\left(
\sigma(\mathbf{x}),
\mathbf{c}(\mathbf{x},\mathbf{d})
\right)
$$

渲染时，沿着相机射线采样多个空间位置，再综合各位置的颜色和密度，得到像素颜色。

在连续体积渲染的经典形式中，像素颜色可以写为：

$$
\mathbf{C}(r)
=
\int_{t_n}^{t_f}
T(t)\sigma(r(t))
\mathbf{c}(r(t),\mathbf{d})\,dt
$$

其中，射线参数化为：

$$
r(t)=\mathbf{o}+t\mathbf{d}
$$

这里：

- $\mathbf{o}$：射线起点；
- $\mathbf{d}$：射线方向；
- $t_n,t_f$：积分区间的起点与终点；
- $\sigma(r(t))$：沿射线的密度；
- $\mathbf{c}(r(t),\mathbf{d})$：沿射线的颜色；
- $T(t)$：光线到达位置 $t$ 之前仍未被吸收的透射率。

透射率可以写为：

$$
T(t)
=
\exp\left(
-\int_{t_n}^{t}\sigma(r(s))\,ds
\right)
$$

直观理解：离相机越近、密度越大的区域，通常越容易影响最终像素颜色；远处的颜色还需要考虑前面介质的透射情况。

实际 NeRF 通常采用离散采样和数值积分近似上述连续公式。

#### 11.2 3D Gaussian Splatting：通过显式高斯基元表示场景

3D Gaussian Splatting 使用一组具有空间范围的三维高斯基元。

单个高斯可以由中心位置、协方差矩阵和其他属性描述。

其空间分布可写为：

$$
G(\mathbf{x})
=
\exp\left(
-\frac{1}{2}
(\mathbf{x}-\boldsymbol{\mu})^T
\boldsymbol{\Sigma}^{-1}
(\mathbf{x}-\boldsymbol{\mu})
\right)
$$

其中：

- $\boldsymbol{\mu}$：高斯中心；
- $\boldsymbol{\Sigma}$：协方差矩阵，决定基元的尺度和方向。

实际 3D Gaussian Splatting 还会为基元学习不透明度、颜色或球谐系数等属性。

渲染时，将三维高斯投影到图像平面，再对可见基元进行排序与混合，生成最终图像。

#### 11.3 两种方法的核心区别

| 比较维度 | NeRF | 3D Gaussian Splatting |
|---|---|---|
| 场景表示 | 连续神经场 | 显式三维高斯基元 |
| 查询方式 | 神经网络预测空间属性 | 访问和投影高斯基元 |
| 主要渲染思路 | 沿射线进行体积渲染 | 投影基元并进行混合 |
| 优势 | 连续表示，适合学习复杂空间函数 | 通常具有较高的渲染效率 |
| 主要代价 | 射线采样和网络查询可能昂贵 | 基元数量、可见性和内存管理需要权衡 |

这里不应简单理解为“NeRF 是隐式的，所以一定慢；3DGS 是显式的，所以一定快”。具体速度和质量仍取决于实现、场景规模、硬件和训练设置。

两者的共同点是：都通过可优化的场景参数生成图像，并利用观测图像的监督信号调整场景表示。

---

### 12. 与 Deep Learning 和 Multi-View Geometry 的联系



#### 12.1 与 深度学习 的联系

在深度学习 中，我们使用反向传播优化神经网络：

$$
\text{输入}
\rightarrow
\text{模型}
\rightarrow
\text{预测结果}
\rightarrow
\text{损失}
$$

在可微渲染中，流程变为：

$$
\text{场景参数}
\rightarrow
\text{渲染器}
\rightarrow
\text{生成图像}
\rightarrow
\text{图像损失}
$$

随后通过反向传播更新场景参数。

两者共享相同的梯度优化原理，但可微渲染需要处理额外的几何、投影、遮挡和采样问题。

#### 12.2 与 Multi-View Geometry 的联系

Multi-View Geometry 主要研究如何利用相机几何与多视图对应关系恢复三维信息。

例如，通过相机投影关系，将三维点映射到二维图像：

$$
\tilde{\mathbf{x}}
\sim
K[R\mid \mathbf{t}]\tilde{\mathbf{X}}
$$

其中：

- $\tilde{\mathbf{X}}$：齐次三维点；
- $\tilde{\mathbf{x}}$：齐次图像点；
- $K$：相机内参矩阵；
- $R,\mathbf{t}$：相机外参；
- $\sim$：表示齐次坐标下的比例相等。

而可微渲染进一步考虑：给定一个三维场景表示，如何通过相机模型生成整张图像，以及如何利用图像误差优化场景。

可以将二者的关系概括为：

- Multi-View Geometry 帮助我们理解相机、投影、对应关系和三维几何；
- Scene Representation 研究如何存储和表达三维世界；
- Differentiable Rendering 研究如何从场景生成图像，并通过梯度进行优化。

它们共同构成现代三维视觉的重要基础。

---

### 13. 总结

本节课最重要的不是记住所有表示方法的名称，而是理解它们背后的设计取舍。

**第一，表示方式决定了场景如何存储、查询和优化。**

网格、点云、体素、SDF、神经场和高斯基元各有适用场景。

**第二，连续表示不意味着必须使用纯 MLP。**

离散网格、插值、自适应结构和神经网络可以组合起来，提高查询效率并保留表达能力。

**第三，可微渲染把三维场景参数与图像监督连接起来。**

其核心仍然是链式法则和梯度优化，但渲染中的可见性、遮挡和采样会带来额外的技术挑战。

**第四，NeRF 和 3D Gaussian Splatting 是理解现代三维场景表示的两个重要例子。**

前者以连续神经场和体积渲染为核心，后者使用显式高斯基元及其投影混合。




### 表征学习与生成建模

#### Introduction to Representation Learning and Generative Modeling

---

#### 0. 本讲的核心问题

计算机视觉模型究竟在学习什么？

假设输入一张图片 $x$，神经网络将它映射为一个特征向量：

$$
z = f_\theta(x), \qquad z \in \mathbb{R}^d
$$

其中：

- $x$：输入图像。
- $f_\theta$：参数为 $\theta$ 的神经网络编码器。
- $z$：图像的特征表示（representation）。
- $d$：特征向量的维度。

本讲主要回答以下问题：

1. 什么样的表征才算好？
2. 没有人工标签时，模型如何学到有意义的特征？
3. 表征学习与生成模型有什么联系？
4. 为什么压缩、去噪和缺失信息补全能够帮助模型学习？
5. CLIP、DINO 和扩散模型分别在学习什么？

**核心思想：** 表征学习寻找适合特定任务的信息编码方式；生成建模则从概率分布的角度描述数据及其不确定性。二者可以相互促进。

---

#### 1. 什么是好的表征？

##### 1.1 从原始图像到特征向量

一张 RGB 图片可以表示为：

$$
x \in \mathbb{R}^{H \times W \times 3}
$$

例如，一张 $224 \times 224$ 的图片包含：

$$
224 \times 224 \times 3 = 150528
$$

个像素值。

直接使用原始像素完成高层语义任务并不容易。两张语义上相同的猫的照片，可能因为位置、姿态、光照、背景和颜色不同，而具有完全不同的像素值。

因此，我们希望将图像转换为更有用的特征：

$$
x \xrightarrow{f_\theta} z
$$

理想情况下，$z$ 能够保留完成目标任务所需要的信息，同时减少无关变化的影响。

例如：

- **分类：** 保留与类别相关的信息。
- **检索：** 让语义相似的图片在特征空间中接近。
- **目标跟踪：** 保留物体身份和外观线索。
- **三维重建：** 保留几何结构、深度和视角相关的信息。
- **图文对齐：** 让图片与对应文字具有可比较的语义表示。

注意：**不存在对所有任务都最优的表征。** 对分类很好的特征，不一定适合精确定位、三维重建或图像生成。

##### 1.2 不变性与等变性

当输入发生变化时，表征应该如何变化？

###### 不变性（Invariance）

如果输入发生某种变化，但任务结果不应改变，那么我们希望表征尽可能保持不变：

$$
f(Tx) \approx f(x)
$$

其中 $T$ 表示某种输入变换。

例如，猫在图片中稍微向左移动，不应该让分类结果从“猫”变成“狗”。

###### 等变性（Equivariance）

如果输入发生变化，输出也应该按照相应规则变化，那么我们希望模型具有等变性：

$$
f(Tx) \approx T'f(x)
$$

其中：

- $T$：输入空间中的变换。
- $T'$：输出空间中对应的变换。

例如，图片中的物体向左移动，分割掩码也应相应向左移动。

两者的区别：

- 分类往往希望具有一定的不变性。
- 检测、分割和跟踪等空间任务通常需要保留几何对应关系。
- 三维视觉需要处理视角变化，同时保留重要的空间信息。

因此，不能简单地认为不变性越强越好。关键在于：**保留任务需要的信息，忽略任务不需要的变化。**

##### 1.3 如何评价表征？

可以从四个角度思考：

1. **任务性能：** 用于分类、检索、分割等任务时，效果如何？
2. **泛化能力：** 能否处理训练集中没有出现过的物体、背景和环境？
3. **鲁棒性：** 对光照、噪声、视角和裁剪等变化是否稳定？
4. **迁移能力：** 在一个数据集上学到的特征，能否迁移到其他任务或数据集？

一种常见的评估方法是冻结预训练编码器，只训练一个线性分类器：

$$
\hat{y} = Wz + b
$$

这称为 **Linear Probe（线性探测）**。

如果只使用一个线性层就能取得较好的效果，说明表征中可能已经包含容易被线性读出的任务相关信息。

但 Linear Probe 不能完整代表表征质量。它主要评估特定任务上的线性可分性，并不能直接衡量所有下游任务、鲁棒性或几何信息。

---

#### 2. 监督学习与自监督表征学习

##### 2.1 传统监督学习

假设训练数据为：

$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N}
$$

其中 $x_i$ 是图像，$y_i$ 是人工标注的类别。

模型通常优化：

$$
\min_\theta
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}(f_\theta(x_i), y_i)
$$

这种方式直接告诉模型什么是正确答案。

优点是目标清晰；缺点是通常需要大量高质量标签，而且标签只覆盖预先定义的任务。

##### 2.2 自监督学习（Self-Supervised Learning）

自监督学习的基本思想是：不一定需要人工标注，可以从数据本身构造训练目标。

例如：

- 从同一张图片生成两个不同视图，让模型学习它们之间的关联。
- 随机遮住图像的一部分，让模型预测被遮挡的信息。
- 给图像加入噪声，让模型学习恢复原始信号。
- 利用视频相邻帧之间的时空关系构造学习目标。

一般形式可以写为：

$$
\min_\theta
\mathbb{E}_{x \sim p_{\text{data}}}
\left[
\mathcal{L}_{\text{pretext}}(x; \theta)
\right]
$$

其中 $\mathcal{L}_{\text{pretext}}$ 是代理任务的损失函数。

关键问题是：

**代理任务学到的能力，是否对真正关心的下游任务有帮助？**

如果代理目标与下游任务关系不大，模型可能学到一些容易利用、却没有多少实际价值的规律。

---

#### 3. 代理任务：压缩、去噪与缺失信息补全

课程大纲特别提到了三个例子：

- Compression：压缩。
- Denoising：去噪。
- Imputation：缺失信息补全。

它们展示了模型如何通过解决间接问题来学习数据结构。

##### 3.1 压缩（Compression）

压缩的目标是用更少的信息表示原始数据：

$$
z = f_\theta(x), \qquad
\hat{x} = g_\phi(z)
$$

其中：

- $z$：压缩后的表示。
- $\hat{x}$：重建结果。

最简单的重建损失是：

$$
\mathcal{L}_{\text{rec}} = \|x - \hat{x}\|_2^2
$$

但是，压缩并不是无损的。编码器必须决定哪些信息值得保留。

例如，一张图片里有一只猫和复杂的草地背景。若压缩得非常厉害，模型可能优先保留大致轮廓，却丢失细小纹理。

这里存在一个重要的权衡：

- 压缩率越高，能够保留的信息通常越少。
- 重建质量越高，往往需要更多的表示容量。
- 对不同任务来说，“重要信息”的定义并不相同。

在信息论中，可以把这个问题联系到率失真理论（Rate-Distortion Theory）：

$$
\min D(x, \hat{x})
\quad \text{subject to} \quad
R \leq R_{\max}
$$

其中：

- $D$：失真程度。
- $R$：编码所需的比特率。
- $R_{\max}$：允许的最大比特率。

**为什么压缩能够帮助表征学习？**

模型必须从大量像素中提取可复用的结构。但单纯的像素重建也可能使模型过分关注纹理或低层细节，不一定自动产生优秀的语义特征。

##### 3.2 去噪（Denoising）

设干净图像为 $x$，加入噪声后得到：

$$
\tilde{x} = x + \epsilon
$$

其中 $\epsilon$ 是噪声。

模型接收 $\tilde{x}$，试图预测原始图像：

$$
\hat{x} = g_\theta(\tilde{x})
$$

一种训练目标是：

$$
\mathcal{L}_{\text{denoise}}
=
\mathbb{E}_{x,\epsilon}
\left[
\|g_\theta(x+\epsilon)-x\|_2^2
\right]
$$

模型需要利用数据中稳定的统计规律来恢复合理的结果。

例如，图像中的噪点可能是随机的，但物体轮廓、边缘和局部结构通常更有规律。

不过，去噪不意味着模型必然学到高层语义。它可能主要学习低层图像统计规律，实际效果取决于噪声类型、网络结构、数据分布和训练目标。

##### 3.3 缺失信息补全（Imputation）

假设图像的一部分不可见。我们希望根据可见区域推断缺失区域。

设 $m$ 是可见区域掩码：

$$
x_{\text{visible}} = m \odot x
$$

其中 $\odot$ 表示逐元素相乘。

模型需要根据可见信息预测缺失内容：

$$
\hat{x}_{\text{missing}}
=
g_\theta(x_{\text{visible}}, m)
$$

例如，遮挡一张狗的照片中的一部分，让模型预测被遮挡的区域。

这个任务要求模型利用可见区域中的上下文信息，可能涉及物体形状、局部纹理、部件关系和场景结构。

但是，补全通常不是唯一答案。例如，只看到人的半张脸时，缺失部分可能存在多种合理结果。因此，我们还需要讨论概率分布和不确定性。

##### 3.4 三种代理任务的对比

| 代理任务 | 模型面对的问题 | 可能学到的规律 | 主要局限 |
|---|---|---|---|
| 压缩 | 如何用更少的信息表示数据？ | 低维结构、冗余关系 | 可能丢掉任务重要的信息 |
| 去噪 | 如何从受扰动的数据恢复信号？ | 稳定结构、数据统计规律 | 可能只学习低层特征 |
| 缺失补全 | 如何根据上下文推断未知部分？ | 局部结构、上下文关系 | 可能存在多个合理答案 |

这三个任务的共同点是：模型并不直接接收“猫”“汽车”等人工语义标签，却可以通过数据本身构造学习信号。

---

#### 4. 生成建模：从预测一个答案到描述可能的答案

##### 4.1 判别模型与生成模型

判别模型通常关注：

$$
p(y \mid x)
$$

即给定图像 $x$，预测标签 $y$ 的概率。

例如，分类器可能输出：

$$
p(y=\text{cat} \mid x) = 0.95
$$

生成模型则通常试图学习数据本身的分布：

$$
p_\theta(x)
$$

或者在给定条件 $c$ 时学习：

$$
p_\theta(x \mid c)
$$

例如，给定文字描述，生成符合描述的图像。

两类模型并不是绝对对立的。生成模型也可以用于判别相关任务，而判别模型也能学习丰富的特征表示。

##### 4.2 概率密度估计（Density Estimation）

现实中的图像来自某种复杂的数据分布 $p_{\text{data}}(x)$。我们通常不知道这个分布的精确形式，只能观察有限的数据样本：

$$
x_1, x_2, \ldots, x_N
\sim p_{\text{data}}(x)
$$

生成建模试图学习一个近似分布 $p_\theta(x)$，使它能够描述真实数据的统计规律。

一种经典目标是最大化对数似然：

$$
\max_\theta
\sum_{i=1}^{N}
\log p_\theta(x_i)
$$

直觉是：模型应该给训练数据中的真实样本分配较高的概率密度。

但这不等于模型一定能生成高质量图片。模型的表示能力、优化过程、数据覆盖范围以及生成算法都会影响最终结果。

还要区分两个概念：

- **概率密度高：** 模型认为该位置附近的数据较为常见。
- **生成质量高：** 生成样本在视觉、语义或应用指标上表现良好。

二者相关，但并不完全等价。

##### 4.3 不确定性建模（Uncertainty Modeling）

如果输入信息不完整，模型可能无法确定唯一答案。

例如，只观察到一只动物的尾巴和部分身体，可能无法判断它究竟是哪种动物。一个合理的模型应该允许多个可能结果，而不是假装自己百分之百确定。

在条件生成中，我们可以写为：

$$
p_\theta
\left(
x_{\text{missing}}
\mid x_{\text{visible}}
\right)
$$

这个分布描述了：给定可见区域后，缺失区域有哪些可能的取值，以及它们相对有多合理。

从分布中采样：

$$
x_{\text{missing}}
\sim
p_\theta
\left(
x_{\text{missing}}
\mid x_{\text{visible}}
\right)
$$

便可能得到多个不同但合理的补全结果。

这也解释了为什么单一的均方误差重建可能有局限：当真实答案存在多个可能性时，直接回归一个平均结果，可能产生模糊或不够自然的图像。

**重要启示：** 对于存在歧义的问题，预测一个答案和描述答案的不确定性，是两种不同的建模目标。

---

#### 5. 表征学习与生成建模有什么关系？

二者关注的问题不同，但可以互相连接。

表征学习希望找到有用的编码：

$$
z = f_\theta(x)
$$

生成建模希望描述数据的分布：

$$
p_\theta(x)
$$

##### 5.1 在潜在空间中生成

先把图像编码成潜在变量 $z$，再从潜在变量重建图像：

$$
x \xrightarrow{f_\theta} z
\xrightarrow{g_\phi} \hat{x}
$$

如果潜在变量具有合适的结构，生成模型就可以尝试在潜在空间中学习或采样，再将结果解码回图像空间。

这类思路出现在自编码器、变分自编码器（VAE）以及潜在扩散模型等方法中，但不同方法对潜在变量和概率分布的定义并不相同。

##### 5.2 通过生成任务学习表征

如果模型需要补全被遮挡的图像区域，就必须利用可见内容中的结构信息。

如果模型需要从噪声中恢复图像，就需要学习数据在不同噪声程度下的规律。

这些目标可能帮助模型学习可复用的特征。但生成效果好并不自动意味着表征适合分类、检索或其他任务；仍然需要针对目标任务进行评估。

##### 5.3 统一视角

1. 数据中存在大量规律与冗余。
2. 模型通过某种训练目标学习这些规律。
3. 学到的内部表示可以用于预测、检索、重建或生成。
4. 训练目标决定模型更倾向保留哪些信息。
5. 最终表征是否有用，取决于它与目标任务之间的关系。

---

#### 6. 联系到 CLIP、DINO 和扩散模型

##### 6.1 CLIP：通过跨模态对比学习建立语义空间

CLIP 使用图像编码器和文本编码器：

$$
z_I = f_\theta(x), \qquad
z_T = g_\phi(t)
$$

通常将特征归一化，再计算图像与文本的余弦相似度：

$$
s(x,t)=
\frac{z_I^\top z_T}
{\|z_I\|_2 \|z_T\|_2}
$$

训练目标会鼓励匹配的图文对具有较高相似度，同时区分不匹配的样本。

CLIP 的重点不只是“识别图片是什么”，而是学习一种能够连接视觉和语言的表征空间。

从本讲角度看：

- **表征是什么？** 图像与文本的向量表示。
- **学习信号是什么？** 图文配对关系及对比目标。
- **表征希望保留什么？** 有助于跨模态语义匹配的信息。
- **可能的局限是什么？** 训练数据和目标会影响模型对细粒度空间信息、长尾概念和分布外样本的表现。

##### 6.2 DINO：从图像自身学习视觉特征

DINO 系列方法属于自监督视觉表征学习的重要方向。

其核心思想是利用不同图像视图之间的关系构造训练信号，而不是依赖每张图像的人工类别标签。

从概念上看：

1. 从同一张图像生成不同视图，例如裁剪或改变尺度。
2. 将视图输入网络。
3. 通过训练目标，使不同视图的输出具有期望的关系。
4. 让网络逐渐学习可迁移的视觉结构。

具体实现会因 DINO 版本而异，不能将所有版本简单归结为同一个损失函数。

DINO 回答了一个关键问题：

**没有人工类别标签，模型能否仅凭数据内部的结构学到有用的视觉特征？**

评估这类表征时，可以考察线性探测、迁移任务，以及特征对物体区域或语义结构的反映。

##### 6.3 扩散模型：通过逐步去噪学习数据分布

扩散模型通常构造一个逐渐向数据加入噪声的过程，再训练模型学习逆向去噪。

一个常见的前向过程写为：

$$
x_t =
\sqrt{\bar{\alpha}_t}x_0
+
\sqrt{1-\bar{\alpha}_t}\epsilon,
\qquad
\epsilon \sim \mathcal{N}(0,I)
$$

其中：

- $x_0$：原始数据。
- $x_t$：第 $t$ 个噪声阶段的数据。
- $\bar{\alpha}_t$：控制原始信号保留比例的系数。
- $\epsilon$：高斯噪声。

常见训练目标是预测噪声：

$$
\mathcal{L}_{\text{diff}}
=
\mathbb{E}_{x_0,t,\epsilon}
\left[
\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2
\right]
$$

训练完成后，模型可以通过逐步去噪生成样本。不同扩散模型可能预测噪声、原始样本、速度参数或其他等价参数化量。

从本讲角度看：

- 去噪是构造学习信号的一种方式。
- 模型通过训练学习数据在不同噪声程度下的统计结构。
- 生成过程体现了模型对数据分布的近似。
- 对于条件生成，模型还需要利用文本、类别或其他条件信息。

需要注意：扩散模型的去噪目标与传统去噪自编码器有联系，但它们的训练过程、噪声设计和生成机制并不完全相同。

---

#### 7. 本讲的概念地图

可以用下面这条主线串起整个讲座：

**数据 $x$**

↓ 通过编码器、代理任务或概率模型进行学习

**表征 $z$ 与数据分布 $p_\theta(x)$**

↓ 用不同目标约束模型学习什么

- **压缩：** 保留有限容量下的重要信息。
- **去噪：** 从扰动中恢复数据规律。
- **补全：** 根据上下文推断缺失信息。
- **对比学习：** 让相关样本在表征空间中具有期望的关系。
- **生成建模：** 描述数据及其可能性。

↓ 最终服务于

**分类、检索、分割、跨模态理解、图像生成、视频生成等任务。**

模型不是凭空“理解”数据，而是在训练目标、数据分布和模型结构共同作用下，学习某种有用的统计规律。

---

#### 8. 建议重点掌握的概念

| 概念 | 需要能回答的问题 |
|---|---|
| Representation | 为什么需要把原始图像编码成特征？ |
| Invariance | 哪些输入变化不应该影响任务结果？ |
| Equivariance | 哪些输入变化应该在输出中体现出来？ |
| Self-supervised learning | 没有人工标签时，学习信号从哪里来？ |
| Surrogate task | 为什么代理任务可能帮助下游任务？ |
| Density estimation | 模型如何学习数据的概率分布？ |
| Uncertainty modeling | 为什么一个问题可能存在多个合理答案？ |
| Generative modeling | 生成模型与判别模型的主要目标有什么区别？ |
| Linear probe | 如何初步评价预训练表征？ |

---

#### 9. 


##### 问题 1

为什么一个在 ImageNet 分类任务上表现很好的表征，不一定适合三维重建？

##### 问题 2

不变性和等变性的区别是什么？请各举一个计算机视觉的例子。

##### 问题 3

为什么只使用像素级重建损失，不一定能得到最适合语义分类的特征？

##### 问题 4

生成模型学习 $p(x)$ 与分类模型学习 $p(y \mid x)$ 有什么区别？

##### 问题 5

为什么图像补全可能需要输出一个分布，而不只是一个确定的图像？

##### 问题 6

CLIP、DINO 和扩散模型分别主要通过什么训练信号学习？

##### 

**1.** 分类更关注区分类别所需的信息；三维重建则需要保留几何、深度、视角等信息。分类表征可能压缩或忽略部分几何细节。

**2.** 不变性意味着某种输入变换不应改变表征，例如轻微平移不应改变物体类别；等变性意味着输出应随输入按对应规则变化，例如图像平移后，分割掩码也应平移。

**3.** 像素级损失强调重建细节，而语义分类强调类别相关信息。低层纹理与像素误差并不总是与高层语义一致。

**4.** $p(x)$ 描述数据本身的分布；$p(y \mid x)$ 描述给定输入后标签的条件分布。前者更直接对应生成建模，后者更直接对应判别预测。

**5.** 同一组可见信息可能对应多个合理的缺失区域。概率分布可以表达这些可能性，而单一预测只能给出一个结果。

**6.** CLIP 主要利用图文匹配关系进行对比学习；DINO 利用不同图像视图之间的自监督信号学习特征；扩散模型通常通过不同噪声阶段的去噪目标学习数据分布。



### Diffusion 1：Flow and Diffusion Models

**课程：** MIT 6.8300 — Advances in Computer Vision  
**核心主题：** 生成建模、ODE/SDE、Flow Models、Flow Matching

#### 1. 从生成问题到概率分布

##### 1.1 如何定义“生成得好”？

假设我们希望模型生成一张符合提示词 `A picture of a dog` 的图片。

一张图片生成得好不好，最初看起来是主观问题。课程引入了一个概率视角：

> 一张图片越符合真实数据的统计规律，它在目标数据分布下出现的可能性就越高。

令 $x$ 表示一张图片，$p_{\text{data}}(x)$ 表示目标数据分布的概率密度。

生成建模的目标可以写成：

$$
x \sim p_{\text{data}}(x)
$$

这里的 $p_{\text{data}}$ 描述我们希望生成的数据具有怎样的分布。对于图像生成任务，它可以理解为真实图像在图像空间中的分布。

需要注意：

- $p_{\text{data}}(x)$ 通常是未知的。
- 我们拥有的是从这个分布采样得到的训练数据，而不是完整的概率密度函数。
- 模型的目标是学习生成符合该分布的新样本，而不是简单地复制训练图片。

对于文本条件生成，更完整的目标是从条件分布中采样：

$$
x \sim p_{\text{data}}(x \mid y)
$$

其中 $y$ 表示文本提示词或其他条件。

##### 1.2 生成模型的基本思想

直接从复杂的数据分布中采样很困难。因此，生成模型通常从一个容易采样的初始分布出发，再将样本转换成目标数据。

$$
z \sim p_0(z)
\quad \xrightarrow{\text{Generative Model}} \quad
x \sim p_{\text{data}}(x)
$$

常见的初始分布是标准高斯分布：

$$
z \sim \mathcal{N}(0,I)
$$

**直觉：** 初始样本可以看作随机噪声。模型逐步改变这个样本，使最终结果符合目标数据的统计规律。

在 Flow 和 Diffusion 模型中，这种转换可以通过连续时间的动力系统来描述。

#### 2. Flow 与 Diffusion：用微分方程描述生成过程

##### 2.1 Flow Model：通过 ODE 生成样本

Flow Model 使用常微分方程（Ordinary Differential Equation，ODE）描述样本随时间的变化：

$$
\frac{dX_t}{dt} = u_t^\theta(X_t)
$$

其中：

- $X_t$：时间 $t$ 的样本状态。
- $t$：连续时间，课程采用从 $t=0$ 到 $t=1$ 的生成过程。
- $u_t^\theta(x)$：神经网络表示的速度场（vector field）。
- $\theta$：神经网络的可学习参数。

速度场告诉我们：当样本位于 $x$、时间为 $t$ 时，下一瞬间应该朝哪个方向移动，以及移动多快。

生成过程为：

1. 从初始分布采样 $X_0 \sim p_0$。
2. 使用神经网络计算当前速度 $u_t^\theta(X_t)$。
3. 根据 ODE 数值积分，逐步更新样本。
4. 得到终点 $X_1$，作为生成结果。

##### 2.2 Euler 方法：如何数值求解 ODE？

实际计算机无法直接执行连续的无限小变化，因此需要使用数值积分方法。

最简单的方法是 Euler 方法：

$$
X_{t+\Delta t}
\approx X_t + \Delta t\,u_t^\theta(X_t)
$$

其中 $\Delta t$ 是时间步长。

**理解公式：**

- $u_t^\theta(X_t)$ 给出当前移动方向和速度。
- $\Delta t\,u_t^\theta(X_t)$ 是这一小段时间内的近似位移。
- 不断重复这个过程，就可以得到从初始噪声到生成样本的轨迹。

若将时间区间 $[0,1]$ 划分为 $N$ 个等长步骤，则：

$$
\Delta t = \frac{1}{N}
$$

$$
X_{k+1}
= X_k + \frac{1}{N}u_{t_k}^\theta(X_k)
$$

这里的 $k$ 是离散计算步骤的编号，不应与连续时间 $t$ 混淆。

一般而言，步长越小，Euler 方法对连续轨迹的近似通常越精细，但计算成本也越高。

##### 2.3 Diffusion Model：通过 SDE 生成样本

Diffusion 模型可以使用随机微分方程（Stochastic Differential Equation，SDE）描述：

$$
dX_t = u_t^\theta(X_t)\,dt + \sigma_t\,dW_t
$$

其中：

- $u_t^\theta(X_t)\,dt$：漂移项，描述有方向的变化。
- $dW_t$：布朗运动的随机增量。
- $\sigma_t$：扩散系数，控制随机扰动的强度。

与 ODE 相比，SDE 在确定性变化之外加入了随机噪声。

**直觉对比：**

- ODE：给定初始状态和速度场，轨迹由动力学确定。
- SDE：除了漂移变化，还存在随机扰动；即使漂移项相同，不同的随机过程也可能产生不同轨迹。

这并不意味着 ODE 只能用于 Flow、SDE 只能用于 Diffusion 的所有实现。它们是本讲介绍的两种核心生成描述方式，后续课程会进一步讨论二者之间的联系。

##### 2.4 ODE 与 SDE 对比

| 对比项 | Flow：ODE | Diffusion：SDE |
|---|---|---|
| 基本方程 | $\frac{dX_t}{dt}=u_t^\theta(X_t)$ | $dX_t=u_t^\theta(X_t)dt+\sigma_t dW_t$ |
| 状态变化 | 由速度场确定 | 由漂移项和随机扰动共同决定 |
| 随机性 | 给定初始状态后，轨迹是确定的 | 演化过程中包含随机性 |
| 数值模拟 | 例如 Euler 积分 | 需要同时模拟漂移与随机增量 |
| 共同目标 | 将初始分布转换成目标数据分布 | 将初始分布转换成目标数据分布 |

##### 2.5 本讲的时间约定

本讲采用以下生成方向：

$$
t=0:\text{Noise}
\quad \longrightarrow \quad
t=1:\text{Data}
$$

也就是说：

- $X_0$：从初始噪声分布采样。
- $X_1$：生成的最终样本。
- $X_t$：生成过程中的中间状态。

**注意：** 不同 Diffusion 文献可能采用不同的时间方向。阅读公式时，应先确认作者如何定义 $t=0$ 和 $t=1$。

#### 3. Flow Matching：如何训练速度场？

##### 3.1 训练目标

Flow Model 的采样过程依赖速度场 $u_t^\theta(x)$，因此训练的关键是让神经网络学会正确的速度场。

但真实数据分布的完整演化规律通常未知，我们也无法直接获得真实的边缘速度场。因此，课程采用 Flow Matching 思路：构造从噪声到数据的概率路径，并利用可计算的监督信号训练神经网络。

##### 3.2 Conditional 与 Marginal

这是 Flow Matching 中非常重要的一组术语。

| 术语 | 含义 | 直观理解 |
|---|---|---|
| Conditional（条件） | 针对一个给定的数据点 | 固定一个样本，研究与它关联的路径 |
| Marginal（边缘） | 跨越整个数据分布 | 考虑不同数据样本共同形成的整体分布 |

设 $z$ 是从真实数据分布采样得到的一个数据点。

对于固定的 $z$，我们可以定义条件概率路径：

$$
p_t(x \mid z)
$$

它描述在给定数据点 $z$ 的情况下，时间 $t$ 的样本状态如何分布。

对所有数据点进行平均，就得到边缘概率路径：

$$
p_t(x)
=
\int p_t(x \mid z)\,p_{\text{data}}(z)\,dz
$$

这条边缘路径描述整个生成过程中，所有样本的总体分布如何随时间变化。

##### 3.3 从条件路径到边缘路径

课程强调：我们希望模型学习的是能够驱动整体分布演化的边缘速度场。

记：

- $u_t(x \mid z)$：条件速度场。
- $u_t(x)$：边缘速度场。

二者的任务不同：

- 条件速度场描述给定数据点 $z$ 时，如何沿对应的条件概率路径运动。
- 边缘速度场描述整体概率分布应如何运动，才能沿目标边缘概率路径演化。

条件路径通常更容易构造，条件速度场也更容易计算。因此，可以使用大量不同数据点对应的条件监督信号来训练神经网络，使其近似边缘速度场。

**关键理解：** 条件速度场与边缘速度场不能简单地视为处处相同。Flow Matching 的核心，是利用可计算的条件目标来学习所需的边缘动力学。

##### 3.4 连续性方程：速度场如何改变概率分布？

为了理解速度场与概率分布之间的关系，需要引入连续性方程（Continuity Equation）：

$$
\frac{\partial p_t(x)}{\partial t}
=
-\nabla \cdot \left(p_t(x)u_t(x)\right)
$$

其中：

- $p_t(x)$：时间 $t$ 的概率密度。
- $u_t(x)$：对应的速度场。
- $\nabla\cdot$：散度运算，用于描述向量场在某一点附近的净流出程度。

**直觉：概率质量守恒。**

可以把概率密度想象成流体的密度，把速度场想象成流体的流速：

- 如果某个区域流出的概率质量多于流入的概率质量，该区域的概率密度就会下降。
- 如果流入多于流出，该区域的概率密度就会上升。

连续性方程刻画的正是这种概率质量随时间的变化规律。

它建立了两个关键对象之间的联系：

$$
\boxed{
\text{速度场}
\quad \longleftrightarrow \quad
\text{概率分布的演化}
}
$$

如果速度场设计得当，沿着该速度场求解 ODE，就能够使样本的边缘分布按照预期的概率路径演化。

##### 3.5 Flow Matching 的训练逻辑

可以把本讲的 Flow Matching 思路整理成以下流程：

1. **采样数据点：** 从真实数据分布采样 $z$。
2. **构造概率路径：** 定义噪声与数据之间的条件概率路径。
3. **获得监督信号：** 计算该条件路径对应的目标速度。
4. **训练神经网络：** 使 $u_t^\theta(x)$ 逼近目标速度。
5. **执行生成：** 从初始噪声出发，使用训练好的边缘速度场求解 ODE。
6. **返回终点：** 得到生成样本 $X_1$。

课程随后以 Meta MovieGen 和 Stable Diffusion 3 为例，说明 Flow Matching 可以用于大规模视频与图像生成。

##### 3.6 为什么不直接用条件速度场生成？

这里容易产生一个误解：既然条件速度场更容易计算，为什么还要学习边缘速度场？

因为生成时通常只有一个随机初始噪声样本，并不知道它应该对应哪个真实数据点 $z$。

因此，模型不能依赖一个预先指定的真实数据点来决定整个生成过程。它需要学习适用于整体分布的速度场，从噪声出发生成合理的数据样本。

条件路径和条件速度场主要提供训练监督；边缘速度场则承担实际生成时的整体分布演化任务。

#### 4. Diffusion 1 总结

##### 4.1 核心知识地图

```text
生成问题
  ↓
将生成理解为从数据分布中采样
  ↓
从容易采样的初始分布出发
  ↓
用连续时间动力系统完成分布转换
  ├── ODE → Flow Model
  └── SDE → Diffusion Model
  ↓
Flow Model 的训练
  ├── 构造条件概率路径
  ├── 获得条件速度监督
  ├── 学习边缘速度场
  └── 使用 ODE 数值积分生成样本
```

##### 4.2 必须掌握的公式

**（1）Flow 的 ODE**

$$
\frac{dX_t}{dt}=u_t^\theta(X_t)
$$

含义：神经网络给出当前样本应当如何移动。

**（2）Euler 数值积分**

$$
X_{t+\Delta t}\approx X_t+\Delta t\,u_t^\theta(X_t)
$$

含义：用一系列小步近似连续生成轨迹。

**（3）Diffusion 的 SDE**

$$
dX_t=u_t^\theta(X_t)\,dt+\sigma_t\,dW_t
$$

含义：漂移项与随机噪声共同决定状态变化。

**（4）连续性方程**

$$
\frac{\partial p_t(x)}{\partial t}
=-\nabla\cdot(p_t(x)u_t(x))
$$

含义：速度场决定概率质量如何在空间中流动。

##### 4.3 复习时要能回答的问题

- 为什么生成建模可以理解为从目标数据分布中采样？
- 为什么模型可以从简单的高斯噪声分布出发？
- ODE 与 SDE 在生成过程中的区别是什么？
- Euler 方法如何近似求解 ODE？
- Conditional 与 Marginal 分别指什么？
- 连续性方程如何将速度场与概率分布的演化联系起来？
- 为什么 Flow Matching 可以利用条件速度来学习边缘速度场？
- 为什么训练时使用条件监督，不代表生成时必须知道对应的真实数据点？

##### 4.4 一句话记忆

**Diffusion 1 的主线：** 将生成看成从噪声分布到数据分布的转换；用 ODE 或 SDE 描述转换过程；对于 Flow Model，则通过构造条件概率路径和条件速度监督，训练能够驱动整体分布演化的速度场。

### D2. Flow and Diffusion Models — 2

#### 本讲学习路线

本讲建立在 D1 的 Flow Matching 基础上，主要回答五个问题：

1. **Score Matching：** 除了学习速度场，为什么还可以学习概率密度的梯度？
2. **Diffusion Sampling：** 如何利用 score function 实现随机采样？
3. **Classifier-Free Guidance（CFG）：** 如何让生成结果更符合文本提示？
4. **Neural Network Architectures：** 为什么生成模型需要 U-Net 或 Diffusion Transformer（DiT）？
5. **Large-scale Models：** Stable Diffusion 3 和 MovieGen 如何应用这些思想？

建议重点掌握前三部分的数学推导，后两部分重点理解设计动机与模型之间的联系。

---

### 1. 从 Flow Matching 到 Score Matching

#### 1.1 回顾：Flow Matching 学习什么？

在 D1 中，我们将生成过程表示为一个随时间变化的速度场：

$$
\frac{dX_t}{dt}=u_t^\theta(X_t)
$$

其中：

- $X_t$：时刻 $t$ 的样本，例如一张正在生成的图像。
- $u_t^\theta(x)$：神经网络预测的速度场。
- $\theta$：神经网络参数。

给定初始噪声 $X_0$，通过求解 ODE，逐步将样本从噪声分布移动到目标数据分布。

Flow Matching 的训练目标可以概括为：让神经网络预测的速度尽可能接近目标速度。

$$
\mathcal{L}_{\mathrm{FM}}
=
\mathbb{E}\left[
\left\|u_t^\theta(X_t)-u_t^*(X_t)\right\|^2
\right]
$$

这里的 $u_t^*$ 表示对应训练构造下的目标速度。实际训练中，我们可以利用条件概率路径构造容易计算的监督信号。

**核心问题：** 速度场不是描述概率分布变化的唯一方式。我们还可以学习一个与概率密度相关的量：Score Function。

#### 1.2 Score Function 的定义

Score Function 定义为对数概率密度关于输入的梯度：

$$
s_t(x)=\nabla_x\log p_t(x)
$$

其中：

- $p_t(x)$：时刻 $t$ 的概率密度。
- $\log p_t(x)$：对数概率密度。
- $\nabla_x$：对输入 $x$ 求梯度。

直觉上，score 描述了概率密度随空间位置变化的方向与强度。

因为

$$
\nabla_x\log p_t(x)
=
\frac{\nabla_x p_t(x)}{p_t(x)}
$$

所以 score 反映的是概率密度的相对变化率，而不只是密度本身的大小。

**注意：** Score 不等于概率密度，也不直接等于速度。它们是不同的数学对象，但在特定的概率路径下，可以建立明确的转换关系。

#### 1.3 一个简单例子：标准高斯分布

设

$$
p(x)=\mathcal{N}(x;0,I)
$$

忽略与 $x$ 无关的常数，其对数概率密度为

$$
\log p(x)=-\frac{1}{2}x^\top x+C
$$

对 $x$ 求梯度：

$$
\nabla_x\log p(x)=-x
$$

因此：

$$
\boxed{s(x)=-x}
$$

这意味着 score 指向高斯分布的高密度区域：当 $x$ 偏离原点时，score 指向原点方向。

对于一般的概率分布，score 通常更加复杂，需要通过神经网络进行学习。

#### 1.4 Conditional 与 Marginal Score

本讲延续 D1 中 Conditional 和 Marginal 的区分。

**Conditional Score：**

$$
s_t(x\mid z)
=
\nabla_x\log p_t(x\mid z)
$$

它描述给定数据样本 $z$ 后，条件概率路径的 score。

**Marginal Score：**

$$
s_t(x)=\nabla_x\log p_t(x)
$$

它描述将所有可能的条件样本综合起来后，边缘概率路径的 score。

二者的区别在于是否显式给定某个数据样本 $z$。

和 Flow Matching 类似，我们希望通过容易计算的条件目标，学习真正需要的边缘 score。

---

### 2. Gaussian Probability Paths：速度场与 Score 的联系

#### 2.1 CondOT 线性概率路径

D1 和本讲都使用一个重要的例子：Conditional Optimal Transport（CondOT）线性路径。

设：

- $\epsilon\sim\mathcal{N}(0,I)$：初始噪声。
- $z\sim p_{\mathrm{data}}$：目标数据样本。

构造路径：

$$
x_t=(1-t)\epsilon+tz
$$

当 $t=0$ 时：

$$
x_0=\epsilon
$$

当 $t=1$ 时：

$$
x_1=z
$$

所以这条路径将噪声与数据通过线性插值连接起来。

对 $t$ 求导：

$$
\frac{dx_t}{dt}=z-\epsilon
$$

因此，沿着这条条件路径的目标速度为

$$
\boxed{u_t(x_t\mid z)=z-\epsilon}
$$

这里有一个很重要的观察：**条件速度是线性路径的导数，因此计算起来非常简单。**

#### 2.2 计算 Conditional Score

给定目标样本 $z$，有

$$
x_t=(1-t)\epsilon+tz
$$

将其改写为

$$
x_t\mid z
\sim
\mathcal{N}\left(tz,(1-t)^2I\right)
$$

对于均值为 $\mu$、协方差为 $\sigma^2 I$ 的高斯分布：

$$
\nabla_x\log p(x)
=
-\frac{x-\mu}{\sigma^2}
$$

因此，这条条件概率路径的 score 为

$$
s_t(x_t\mid z)
=
-\frac{x_t-tz}{(1-t)^2}
$$

由于

$$
x_t-tz=(1-t)\epsilon
$$

代入后得到

$$
\boxed{
s_t(x_t\mid z)
=
-\frac{\epsilon}{1-t}
}
\qquad t<1
$$

这个公式是本讲的重要结果。

它表明：在高斯概率路径下，Conditional Score 和 Conditional Vector Field 都可以用简单的线性表达式计算。

需要注意，$t=1$ 时公式的分母为零，因此该表达式不能直接在端点 $t=1$ 使用。

#### 2.3 为什么早期 Diffusion Models 学习 Score？

对于 CondOT 路径，我们已经得到：

$$
u_t(x_t\mid z)=z-\epsilon
$$

以及

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

因此，可以从 score 中恢复噪声：

$$
\epsilon=-(1-t)s_t(x_t\mid z)
$$

再代入速度公式：

$$
u_t(x_t\mid z)
=
z+(1-t)s_t(x_t\mid z)
$$

这说明，在该条件高斯路径下，速度与 score 存在明确的转换关系。

不过，上式使用的是条件 score。实际生成时，我们通常关心的是边缘速度场和边缘 score。二者之间的转换需要使用对应的边缘概率路径及其参数化公式，不能简单地将所有条件量直接替换成边缘量。

**本讲的核心结论：** 对适当的高斯概率路径，学习边缘 Vector Field 与学习边缘 Score 可以相互转换。因此，Flow Matching 和 Score-based Diffusion 虽然采用不同的训练目标，但在特定条件下具有紧密的数学联系。

---

### 3. Denoising Score Matching（DSM）

#### 3.1 为什么不直接学习边缘 Score？

目标是学习

$$
s_t(x)=\nabla_x\log p_t(x)
$$

但是，真实边缘概率密度 $p_t(x)$ 通常很难直接计算，因此也难以直接计算它的 score。

一种解决办法是：利用已知的条件概率路径，训练模型预测 Conditional Score，再通过条件样本的训练目标学习 Marginal Score。

这就是 Denoising Score Matching（DSM）的基本思路。

#### 3.2 将 Score Matching 转换成噪声预测

对于线性高斯路径：

$$
x_t=tz+(1-t)\epsilon
$$

其中：

$$
\epsilon\sim\mathcal{N}(0,I)
$$

我们已经推导出：

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

因此，如果神经网络预测噪声 $\epsilon$，就可以构造 score 的预测值：

$$
s_t^\theta(x_t)
=
-\frac{\epsilon_\theta(x_t,t)}{1-t}
$$

这里的 $\epsilon_\theta$ 表示网络预测的噪声。

这解释了为什么一些 Diffusion Models 的训练可以被理解为噪声预测：网络需要估计被加入样本中的噪声，从而恢复对应的 score 信息。

#### 3.3 DSM 的训练目标

一种概念性的条件 score matching 目标为

$$
\mathcal{L}_{\mathrm{DSM}}
=
\mathbb{E}_{t,z,x_t}
\left[
\left\|
s_t^\theta(x_t)
-
s_t(x_t\mid z)
\right\|^2
\right]
$$

这里：

- $z$ 是从数据分布中采样的数据。
- $x_t$ 是从对应条件概率路径采样得到的扰动样本。
- $s_t(x_t\mid z)$ 是可计算的条件 score。
- $s_t^\theta(x_t)$ 是模型预测的 score。

对于前面给出的线性高斯路径，将目标 score 代入：

$$
\mathcal{L}_{\mathrm{DSM}}
=
\mathbb{E}
\left[
\left\|
s_t^\theta(x_t)+\frac{\epsilon}{1-t}
\right\|^2
\right]
$$

不同实现可以采用不同的权重或等价的预测参数化；这里的表达式用于展示基本原理。

**需要记住的逻辑：**

1. 对数据加入噪声，构造 $x_t$。
2. 根据已知的条件分布，计算条件 score。
3. 训练神经网络预测该 score，或者预测与之等价的噪声。
4. 通过这种训练方式学习生成所需的边缘 score。

#### 3.4 数值稳定性

由于

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

当 $t$ 接近 $1$ 时，分母变小，score 的数值可能变大。

这会带来数值稳定性方面的挑战。因此，训练时的时间采样、损失加权以及预测参数化都可能影响实际训练效果。

**学习重点：** 理解为什么预测噪声能够学习 score，比死记某个具体损失函数更重要。

---

### 4. Fokker–Planck Equation：SDE 如何改变概率分布？

#### 4.1 从 ODE 走向 SDE

D1 中，Flow 使用 ODE：

$$
dX_t=u_t(X_t)\,dt
$$

Diffusion 则可以使用 SDE：

$$
dX_t=u_t(X_t)\,dt+\sigma_t\,dW_t
$$

其中：

- $u_t(X_t)\,dt$：漂移项，决定平均运动趋势。
- $\sigma_t$：噪声强度。
- $dW_t$：布朗运动增量。

ODE 只包含确定性的动力学；SDE 额外加入随机扰动。

但这引出了一个问题：**如果单个样本在随机运动，那么整个样本群体的概率密度会怎样变化？**

Fokker–Planck Equation 正是描述这个问题的方程。

#### 4.2 Fokker–Planck 方程

对于以下 SDE：

$$
dX_t=u_t(X_t)\,dt+\sigma_t\,dW_t
$$

在标准假设下，其概率密度 $p_t(x)$ 满足

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
=
-\nabla\cdot\left(p_t(x)u_t(x)\right)
+
\frac{\sigma_t^2}{2}\Delta p_t(x)
}
$$

其中：

- $\frac{\partial p_t}{\partial t}$：某个位置处的概率密度随时间的变化。
- $-\nabla\cdot(p_tu_t)$：由漂移速度造成的概率质量输运。
- $\frac{\sigma_t^2}{2}\Delta p_t$：由随机扩散造成的概率质量重新分布。
- $\Delta$：空间 Laplacian 算子。

该表达式假设噪声系数 $\sigma_t$ 只依赖时间，并且在各个空间维度上具有相同的强度。

#### 4.3 与 Continuity Equation 的比较

在 D1 中，我们介绍过连续性方程：

$$
\frac{\partial p_t(x)}{\partial t}
=
-\nabla\cdot\left(p_t(x)u_t(x)\right)
$$

它描述概率质量在速度场作用下的输运。

与之相比，Fokker–Planck 方程多出一项：

$$
\frac{\sigma_t^2}{2}\Delta p_t(x)
$$

这项描述随机扩散带来的变化。

当 $\sigma_t=0$ 时，Fokker–Planck 方程退化为连续性方程。

因此，可以将两者的区别总结为：

- **Continuity Equation：** 概率质量沿速度场输运。
- **Fokker–Planck Equation：** 概率质量既受到速度场输运，也受到随机扩散影响。

#### 4.4 为什么这一点对 Diffusion 很重要？

对于 SDE，即使知道漂移场 $u_t$，样本群体的概率密度也不只由漂移项决定，随机扩散项同样会影响分布。

这意味着：如果希望通过随机过程采样，同时又希望保持某条预先指定的概率路径，就需要协调漂移和扩散之间的关系。

这引出了下一节的 SDE Extension Trick。

---

### 5. SDE Extension Trick：加入随机性，同时保持目标概率路径

#### 5.1 核心思想

假设我们已经有一个目标概率路径 $p_t(x)$，并且它满足连续性方程：

$$
\frac{\partial p_t(x)}{\partial t}
=
-\nabla\cdot\left(p_t(x)u_t(x)\right)
$$

我们希望构造一个具有随机性的 SDE，但又让它的边缘概率密度仍然沿着这条目标路径演化。

课件给出的构造为：

$$
\boxed{
dX_t
=
\left[
u_t(X_t)
+
\frac{\sigma_t^2}{2}
\nabla\log p_t(X_t)
\right]dt
+
\sigma_t\,dW_t
}
$$

注意，这里漂移项不再只是 $u_t$，还增加了一个与 score 有关的修正项：

$$
\frac{\sigma_t^2}{2}\nabla\log p_t(X_t)
$$

#### 5.2 为什么要添加 Score Correction？

先定义修正后的漂移：

$$
b_t(x)
=
u_t(x)
+
\frac{\sigma_t^2}{2}\nabla\log p_t(x)
$$

代入 Fokker–Planck 方程：

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tb_t)
+
\frac{\sigma_t^2}{2}\Delta p_t
$$

展开漂移项：

$$
\begin{aligned}
\frac{\partial p_t}{\partial t}
={}&-\nabla\cdot(p_tu_t)\\
&-\nabla\cdot
\left(
\frac{\sigma_t^2}{2}
p_t\nabla\log p_t
\right)\\
&+\frac{\sigma_t^2}{2}\Delta p_t
\end{aligned}
$$

利用恒等式

$$
p_t\nabla\log p_t=\nabla p_t
$$

得到

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tu_t)
-\frac{\sigma_t^2}{2}\Delta p_t
+\frac{\sigma_t^2}{2}\Delta p_t
$$

两个扩散相关的项相互抵消，因此：

$$
\boxed{
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tu_t)
}
$$

这正是原来的连续性方程。

以上推导假设 $\sigma_t$ 只依赖时间，并且密度与向量场满足所需的正则性条件。

#### 5.3 直觉理解

可以把整个过程理解为三部分：

1. $u_t$ 负责沿着目标速度场移动。
2. 随机噪声 $\sigma_t\,dW_t$ 使采样过程具有随机性。
3. Score correction 对漂移进行修正，抵消随机扩散在边缘密度演化中引入的额外变化。

因此，在适当条件下，可以构造一个**单个样本随机运动，但整体边缘分布仍遵循指定概率路径**的 SDE。

#### 5.4 使用 Score Network 进行随机采样

实际应用中，真实的 score 往往未知，可以用神经网络近似：

$$
s_t^\theta(x)\approx\nabla_x\log p_t(x)
$$

于是采样时可以采用近似的 SDE：

$$
dX_t
=
\left[
u_t(X_t)+\frac{\sigma_t^2}{2}s_t^\theta(X_t)
\right]dt
+
\sigma_t\,dW_t
$$

需要注意，使用近似 score 时，抵消关系不再保证精确成立；采样效果取决于 score 估计的准确性以及数值积分方法。

#### 5.5 Deterministic Sampling 与 Stochastic Sampling

| 比较维度 | 确定性采样 | 随机采样 |
|---|---|---|
| 典型形式 | ODE | SDE |
| 是否显式加入随机增量 | 否 | 是 |
| 主要动力学 | 速度场 | 漂移、扩散及其修正 |
| 本讲中的关键对象 | Vector Field | Score Function 与 SDE |

两种方法并非简单地对应“好”和“坏”。它们提供了不同的采样机制，实际表现取决于模型、概率路径和采样算法。

---

### 6. Classifier-Free Guidance（CFG）

#### 6.1 为什么需要 Guidance？

在文本条件生成中，我们希望模型不仅生成合理的图像，还能生成**符合指定提示词的图像**。

例如：

- 无条件提示：生成一张图像。
- 条件提示：生成一张猫正在烤蛋糕的图像。

即使生成模型能够生成自然的图像，也不代表每张图像都能准确遵循提示词。

Guidance 的目标是增强生成过程对条件信息的响应。

#### 6.2 CFG 的核心公式

记：

- $u_{\mathrm{uncond}}$：无条件预测的速度场。
- $u_{\mathrm{cond}}$：带有条件提示的速度场。
- $w$：Guidance Scale。

课件使用的 CFG 公式为

$$
\boxed{
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w\left(
u_{\mathrm{cond}}-u_{\mathrm{uncond}}
\right)
}
$$

将公式展开：

$$
u_{\mathrm{CFG}}
=
(1-w)u_{\mathrm{uncond}}
+
wu_{\mathrm{cond}}
$$

其中

$$
u_{\mathrm{cond}}-u_{\mathrm{uncond}}
$$

表示条件提示带来的预测差异。

CFG 会沿着这个差异方向对预测进行调整，从而增强条件提示的影响。

#### 6.3 如何理解 Guidance Scale？

**当 $w=0$ 时：**

$$
u_{\mathrm{CFG}}=u_{\mathrm{uncond}}
$$

模型只使用无条件预测。

**当 $w=1$ 时：**

$$
u_{\mathrm{CFG}}=u_{\mathrm{cond}}
$$

得到普通的条件预测，没有额外的 CFG 放大。

**当 $w>1$ 时：**

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{cond}}
+
(w-1)
\left(
u_{\mathrm{cond}}-u_{\mathrm{uncond}}
\right)
$$

此时，条件预测与无条件预测之间的差异被进一步放大。

直觉上，模型会更加积极地沿着有助于满足提示词的方向更新样本。

#### 6.4 CFG 训练：为什么需要空条件？

CFG 的关键设计是：同一个模型既要能处理有条件输入，也要能处理无条件输入。

训练时，随机丢弃一部分条件标签或提示词，将它们替换为空条件。

这样，模型就能学习两种预测：

$$
u_\theta(x_t,t,y)
$$

以及

$$
u_\theta(x_t,t,\varnothing)
$$

其中：

- $y$：正常的文本条件。
- $\varnothing$：空条件。

在采样阶段，通过分别计算条件预测和无条件预测，就可以组合得到 CFG 速度场：

$$
\begin{aligned}
u_{\mathrm{CFG}}
={}&u_\theta(x_t,t,\varnothing)\\
&+w\left[
u_\theta(x_t,t,y)
-u_\theta(x_t,t,\varnothing)
\right]
\end{aligned}
$$

这就是 Classifier-Free Guidance 名称的由来：它不需要额外训练一个独立的分类器来提供 guidance，而是利用同一生成模型的条件和无条件预测。

#### 6.5 Guidance Scale 越大越好吗？

不一定。

随着 $w$ 增大，模型可能更强烈地遵循提示词，但也可能牺牲图像的自然度、多样性或其他质量指标。

课件特别强调：CFG 是一种主要由经验效果支持的实用技巧。使用 $w>1$ 时，得到的速度场通常不再对应原始条件分布的普通预测。

因此，CFG 并不等价于严格地从原始数据分布采样。

**需要掌握的结论：**

- $w=1$：普通条件预测。
- $w>1$：放大条件预测与无条件预测之间的差异。
- 更强的 guidance 不保证整体生成质量一定更好。

---

### 7. 神经网络架构：U-Net 与 Diffusion Transformer

#### 7.1 生成模型对神经网络有什么要求？

本讲将生成模型中的神经网络理解为一个向量场或相关预测函数。

以条件图像生成为例，网络需要接收：

- 时间 $t$。
- 当前带噪图像或中间状态 $x_t$。
- 条件信息 $y$，例如文本提示。

网络输出通常需要与输入图像或潜变量具有相同的形状：

$$
u_\theta(x_t,t,y)\in\mathbb{R}^{\mathrm{shape}(x_t)}
$$

具体输出代表速度、噪声或其他等价的预测目标，取决于模型采用的参数化方式。

因此，普通的图像分类网络不能直接替代生成模型所需要的预测网络：分类器通常输出类别分数，而生成模型需要输出与当前状态同形状的张量。

#### 7.2 U-Net

U-Net 最初用于图像分割，之后被广泛用于 Diffusion Models。

其基本结构包含两部分：

- **Encoder：** 逐渐降低空间分辨率，同时增加通道数。
- **Decoder：** 逐渐恢复空间分辨率。
- **Skip Connections：** 将编码阶段的信息传递给对应的解码阶段。

这种 U 形结构既能提取较大范围的上下文，也能保留较精细的空间信息。

在生成模型中，还需要让网络利用时间 $t$ 和条件提示 $y$，例如通过时间嵌入、条件嵌入等机制。

#### 7.3 Diffusion Transformer（DiT）

DiT 的核心思路是将图像划分成多个 patches，并通过 Transformer 在这些 patches 之间进行注意力计算。

其基本思路来自 Vision Transformer（ViT）。

与 U-Net 的卷积结构不同，DiT 使用 Transformer 处理图像表示，并将时间与条件信息纳入生成网络。

可以从以下角度理解两者：

| 比较维度 | U-Net | DiT |
|---|---|---|
| 主要结构 | 卷积编码器—解码器 | Transformer |
| 主要处理方式 | 多尺度卷积特征 | Patch 表示与注意力 |
| 结构特点 | U 形结构及跳跃连接 | 在 token / patch 表示之间进行注意力计算 |
| 生成任务中的角色 | 预测速度、噪声等 | 同样预测速度、噪声等 |

两者都可以作为生成模型的骨干网络；选择哪一种取决于任务、计算资源和模型设计。

#### 7.4 Latent Diffusion：为什么在潜空间中生成？

直接在高分辨率图像上运行生成模型，计算成本很高；对于视频，额外的时间维度会进一步增加成本。

Latent Diffusion 的思路是：先使用预训练的（变分）自编码器，将图像编码到潜空间，再在潜变量上训练和运行生成模型。

概念上可以表示为：

$$
x \xrightarrow{\mathrm{Encoder}} z
$$

在潜空间中进行生成：

$$
z_T \longrightarrow z_0
$$

再通过解码器恢复图像：

$$
z \xrightarrow{\mathrm{Decoder}} x
$$

这里的时间方向取决于具体模型采用的约定；该示意图仅表达从噪声潜变量到目标潜变量的生成过程。

潜空间通常比原始图像空间具有更低的空间维度，因此能显著降低生成过程的计算开销。

**重点：** Latent Diffusion 改变的是生成发生的表示空间，不是 Flow Matching 或 Score Matching 的基本数学目标。

---

### 8. 大规模模型案例

#### 8.1 Stable Diffusion 3（SD3）

课件给出的 SD3 案例包含以下设计：

- 使用 Flow Matching。
- 采用 CondOT 风格的直线路径。
- 在预训练 VAE 的潜空间中进行生成。
- 使用 Classifier-Free Guidance。
- 使用 MM-DiT 架构。
- 课件示例规模约为 8B 参数。
- 课件示例采用约 50 个采样步骤。
- 课件提及 LAION 数据集作为数据示例。

其中，MM-DiT 是面向多模态信息设计的 Diffusion Transformer 架构。

可以将 SD3 的生成过程理解为：

1. 从潜空间噪声开始。
2. 将当前潜变量、时间与文本条件输入生成网络。
3. 预测当前状态的更新方向。
4. 根据采样算法逐步更新潜变量。
5. 通过 CFG 增强文本条件的影响。
6. 将生成的潜变量解码为图像。

这里的参数量和采样步数是课件案例中的数值，并不意味着所有配置或版本都具有完全相同的设置。

#### 8.2 Meta MovieGen

课件还介绍了 MovieGen 作为视频生成案例：

- 使用 Flow Matching。
- 使用 CondOT 风格的直线路径。
- 在潜空间中进行生成。
- 使用适配视频任务的 DiT 架构。
- 使用 Classifier-Free Guidance。
- 课件给出的模型规模约为 30B 参数。
- 课件提到训练使用 6,144 张 H100 GPU。

视频生成比静态图像生成更具挑战，因为模型不仅需要处理空间信息，还要处理时间维度上的连续性和一致性。

潜空间生成对于视频尤其重要：如果直接处理所有高分辨率视频帧，计算成本会非常高。

#### 8.3 这两个案例说明了什么？

大型生成模型不是单靠某一个技巧实现的，而是多种设计组合的结果：

- **Flow Matching：** 提供训练目标与生成动力学。
- **潜空间表示：** 降低生成计算的开销。
- **DiT / MM-DiT：** 处理复杂的图像或多模态信息。
- **CFG：** 增强生成结果对条件提示的响应。
- **数值采样：** 将模型预测转化为逐步生成的过程。

理解这些模块分别解决什么问题，比单纯记忆模型名称更重要。

---

### 9. Diffusion 文献地图：不同方法之间的关系

#### 9.1 不同论文可能使用不同的时间约定

阅读 Flow 和 Diffusion 文献时，一个常见困难是：不同方法对时间方向的定义并不一致。

**Flow Matching 常见约定：**

$$
t=0:\ \text{Noise}
\qquad
t=1:\ \text{Data}
$$

**Score-based Diffusion 的常见约定：**

$$
t=0:\ \text{Data}
\qquad
t\rightarrow\infty:\ \text{Noise}
$$

这两种时间约定方向相反。

因此，在比较公式之前，首先要确认：时间增加时，样本是在逐渐接近数据，还是逐渐变成噪声。

不要仅凭变量都叫 $t$，就认为两篇论文中的时间具有相同含义。

#### 9.2 连续时间与离散时间

课件还区分了连续时间和离散时间的建模方式。

连续时间方法可以使用 ODE 或 SDE 描述样本的演化。

离散时间方法则可以通过 Markov Chain 描述一系列随机状态之间的转移。

以经典方法为例：

- **DDPM：** 使用离散时间的扩散 Markov Chain。
- **DDIM：** 提供另一种采样方式，并与 Probability Flow ODE 有密切联系。

课件使用近似关系来概括 DDIM 与 Probability Flow ODE 的联系。更严格地说，具体关系取决于所采用的模型和采样设置，不应把所有 DDIM 过程都视为与任意 Probability Flow ODE 完全相同。

#### 9.3 不同的 Noising Procedures

不同生成方法还可能采用不同的方式构造从数据到噪声、或从噪声到数据的概率路径。

例如：

- Flow Matching / Rectified Flow：直接指定概率路径或插值过程。
- Denoising Diffusion Models：通常从一个逐步加噪的过程出发。
- Score-based Diffusion：学习 score，并通过对应的随机或确定性动力学进行采样。

这些方法的公式看起来可能不同，但在特定高斯概率路径及相应条件下，可以建立数学联系。

需要避免一个过度概括：**并不是任意两种扩散过程在所有设置下都完全等价。** 等价性需要依赖具体的路径、时间参数化和动力学条件。

#### 9.4 Flow Matching 为什么灵活？

课件强调，Flow Matching 可以用于将不同的源分布转换为不同的目标分布。

例如：

- 无音频视频 → 有音频视频。
- 低分辨率图像 → 高分辨率图像。
- 未受扰动的细胞 → 受扰动的细胞。

这些例子说明，Flow Matching 不只适用于传统的“噪声生成图像”。

从更一般的角度看，只要能够定义适当的源分布、目标分布以及连接两者的概率路径，就可以考虑学习对应的速度场。

这也是 Flow Matching 在生成建模之外具有应用潜力的原因。

---

### 10. 总结

#### 10.1 概念关系

**第一层：概率路径**

$$
p_t(x)
$$

描述不同时间的样本分布。

**第二层：Vector Field**

$$
u_t(x)
$$

描述样本沿着怎样的方向移动，并通过连续性方程描述概率质量的输运。

**第三层：Score Function**

$$
s_t(x)=\nabla_x\log p_t(x)
$$

描述概率密度的对数梯度。在适当的高斯概率路径下，它可以与速度场相互转换。

**第四层：随机动力学**

$$
dX_t=u_t(X_t)\,dt+\sigma_t\,dW_t
$$

通过 SDE 引入随机性；Fokker–Planck 方程描述相应的边缘密度演化。

**第五层：采样修正**

$$
u_t(x)+\frac{\sigma_t^2}{2}s_t(x)
$$

在本讲讨论的条件下，利用 score 修正漂移，使加入随机噪声的 SDE 仍可沿着目标概率路径演化。

**第六层：条件控制**

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w(u_{\mathrm{cond}}-u_{\mathrm{uncond}})
$$

CFG 通过放大条件预测与无条件预测之间的差异，提高生成过程对条件提示的响应。

#### 10.2 公式



**1. Score Function**

$$
s_t(x)=\nabla_x\log p_t(x)
$$

**2. CondOT 线性路径**

$$
x_t=(1-t)\epsilon+tz
$$

**3. CondOT 条件速度**

$$
u_t(x_t\mid z)=z-\epsilon
$$

**4. CondOT 条件 Score**

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

**5. Fokker–Planck Equation**

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tu_t)
+
\frac{\sigma_t^2}{2}\Delta p_t
$$

**6. Classifier-Free Guidance**

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w(u_{\mathrm{cond}}-u_{\mathrm{uncond}})
$$




**Flow Matching 学习速度场，Score Matching 学习对数概率密度的梯度；对于适当的高斯概率路径，两者可以相互联系。利用 score，还能构造与目标概率路径相容的随机采样过程。CFG 进一步控制生成结果对提示词的响应，而 U-Net、DiT 和潜空间生成则为这些数学方法提供了可扩展的工程实现。**